# 📊 Annual-Report RAG — 15 Technology Parameters (tuned for 8 GB RAM, CPU-only laptop)

**Flow:** company list (CSV) → match each name to its PDF in `annual_reports/` → parse → chunk → *keep only technology-related chunks* → embed (small model, CPU) → hybrid retrieve + rerank → **gpt-oss-120b on Groq (cloud)** writes cited bullets → **one Excel** for all 15 parameters; run in ranges (START/END) and each range is **appended** to the same Excel.

Tuned for: Intel i5-1240P · 8 GB RAM · no NVIDIA GPU. The LLM runs in the cloud, so your laptop only does PDF parsing + embeddings.

## 0. One-time setup (see README.md in the zip)
```bash
python -m venv .venv
.venv\Scripts\activate
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements.txt
```
Then create a free key at console.groq.com and either paste it in Section 9b when asked, or save it in a file `groq_key.txt` next to this notebook.
Put all annual-report PDFs in `annual_reports/` (text PDFs, not scans). Close Chrome/Teams etc. while it runs — 8 GB RAM is tight.

## 1. Check that Python can see your GPU

In [4]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")      # avoids the VS Code "widgets" popup
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import torch

torch.set_num_threads(min(8, os.cpu_count() or 4))
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("Running on CPU - this is expected on your laptop; the notebook is tuned for CPU + 8 GB RAM.")

PyTorch version : 2.14.1+cpu
CUDA available  : False
Running on CPU - this is expected on your laptop; the notebook is tuned for CPU + 8 GB RAM.


## 2. Imports & configuration
All tunable settings are in `CFG`. The defaults are good for a typical 100-300 page annual report.

In [5]:
import json
import os
import re
import time
import urllib.error
import urllib.request
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable

import numpy as np
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

CFG = dict(
    chunk_size=1500,          # characters per chunk (whole paragraphs preferred)
    chunk_overlap=250,        # character overlap when a long paragraph must be split
    min_chunk_chars=30,       # paragraphs shorter than this are treated as noise (page numbers etc.)
    dense_top_n=40,           # candidates from the dense retriever, per query variant
    bm25_top_n=40,            # candidates from BM25, per query variant
    rerank_candidates=40,    # fused pool size sent to the cross-encoder
    final_top_k=10,           # chunks kept for the answer
    max_context_chars=16000,  # context budget sent to the LLM (~4000 tokens)
    embed_model="BAAI/bge-small-en-v1.5",                       # try "BAAI/bge-large-en-v1.5" for higher quality
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",     # try "BAAI/bge-reranker-base" for higher quality
    neighbor_expand=True,     # pull neighbouring chunks of the same page for continuity
    ollama_num_ctx=8192,      # LLM context window (8192 fits comfortably in 8 GB VRAM)
)

STOP = set("""a an the and or of to in for on with by as at from is are was were be been being this that these those
it its their our your we they he she them his her not no nor but if then than so such can could may might will would
shall should do does did done have has had having i you us also which who whom whose what when where why how all any
both each few more most other some only own same too very s t just don now page annual report limited ltd""".split())

SYNONYMS = {
    "ai": ["artificial intelligence", "machine learning", "deep learning", "generative ai", "genai",
           "ai/ml", "data science", "intelligent automation", "copilot", "large language model", "llm"],
    "cloud": ["cloud computing", "cloud migration", "public cloud", "hybrid cloud", "azure", "aws",
              "amazon web services", "google cloud", "data centre", "data center", "saas", "iaas", "paas"],
    "cyber": ["cybersecurity", "cyber security", "information security", "infosec", "zero trust",
              "soc", "security operations centre", "iso 27001", "soc 2", "ransomware", "phishing",
              "vulnerability", "penetration testing", "data privacy", "data protection"],
    "vendor": ["partner", "partnership", "strategic alliance", "collaboration", "joint venture",
               "ecosystem", "vendor", "supplier", "mou", "memorandum of understanding", "tie-up",
               "agreement", "reseller", "system integrator", "oem"],
    "capex": ["capital expenditure", "capex", "purchase of property plant and equipment",
              "additions to property plant and equipment", "capital outlay", "capital spend",
              "investment in technology", "technology investment"],
    "rnd": ["research and development", "r&d", "r & d", "r&d expenses", "research development expenditure",
            "innovation spend", "product development expense"],
    "acquisition": ["acquisition", "acquire", "acquired", "merger", "stake purchase", "business combination",
                    "investment in subsidiary", "buyout"],
    "emerging": ["emerging technology", "next generation technology", "new technology", "innovation",
                 "poc", "proof of concept", "pilot", "blockchain", "internet of things", "iot",
                 "digital twin", "quantum", "edge computing", "5g", "web3", "metaverse", "industry 4.0"],
    "platform": ["platform", "system", "software", "tool", "erp", "sap", "salesforce", "servicenow",
                 "crm", "core banking", "data lake", "data platform", "architecture", "application"],
    "talent": ["talent", "workforce", "upskilling", "reskilling", "training", "learning and development",
               "certification", "capability building", "attrition", "employee development", "academy"],
    "risk": ["risk", "threat", "challenge", "breach", "outage", "downtime", "obsolescence",
             "concentration risk", "third party risk", "regulatory risk", "dependency"],
    "investment": ["investment", "expenditure", "spend", "outlay", "allocation", "budget", "commitment"],
}

SYN_LOOKUP: dict = {}
for _head, _terms in SYNONYMS.items():
    for _t in _terms:
        SYN_LOOKUP.setdefault(_t, _terms)
    SYN_LOOKUP.setdefault(_head, _terms)

THE_14_QUESTIONS = [
    "What AI initiatives has the company undertaken in the past year?",
    "Who are the vendor or strategic partners collaborating with the company on AI projects?",
    "What cloud transformation initiatives are highlighted, and which vendors are involved?",
    "What cybersecurity programs or partnerships are mentioned in the report?",
    "Which emerging technologies are being explored or adopted by the company?",
    "How does the company describe its innovation pipeline or future technology roadmap?",
    "What investments has the company made in technology-related fields (AI, cloud, cybersecurity, digital platforms)?",
    "How much capital expenditure or R&D spending is allocated to technology initiatives?",
    "Are there acquisitions or partnerships specifically aimed at strengthening technology capabilities?",
    "How does the company describe its overall technology landscape?",
    "What are the key platforms, systems, and tools currently in use?",
    "What risks or challenges related to technology are disclosed in the annual report?",
    "How does the company's technology strategy align with its broader business goals?",
    "What talent or workforce development programs are mentioned to support technology adoption?",
]

QUESTION_HINTS = {  # extra lexical hints per question (used for query expansion)
    1: ["ai", "initiatives", "copilot", "generative"],
    2: ["ai", "vendor", "partner"],
    3: ["cloud", "vendor", "partner"],
    4: ["cyber", "programme", "partnership"],
    5: ["emerging"],
    6: ["emerging", "investment", "platform"],
    7: ["investment", "capex", "rnd", "ai", "cloud", "cyber"],
    8: ["capex", "rnd"],
    9: ["acquisition", "vendor", "ai"],
    10: ["platform", "investment", "emerging", "ai", "cloud", "cyber"],
    11: ["platform", "system", "tool"],
    12: ["risk", "cyber", "challenge"],
    13: ["investment", "platform", "strategy"],
    14: ["talent"],
}

# ---- technology keyword filter (used to skip irrelevant pages/chunks on a small laptop); now covers all 11 sectors ----
TECH_RE = re.compile(r"(?<![a-z0-9])(?:ai|artificial intelligence|machine learning|deep learning|generative|genai|llm|"
    r"technolog\w*|tech|digital\w*|cyber\w*|cloud|automat\w*|rpa|bots?|chatbot\w*|analytics|data (?:science|centre|center|lake|platform|privacy|protection|governance|management|warehouse)|"
    r"software|computer\w*|intangible\w*|information (?:technology|security)|it (?:systems?|infrastructure|security|strategy|governance|expenditure)|fintech\w*|apis?|blockchain|iot|iiot|internet of things|"
    r"quantum|robot\w*|drones?|r&d|r & d|research and development|innovat\w*|patents?|capex|capital expenditure|platform\w*|core banking|erp|crm|"
    r"technology absorption|upskill\w*|reskill\w*|certification\w*|academy|legacy (?:system|technology)|outage|encryption|ransomware|phishing|"
    r"aws|azure|google cloud|microsoft|oracle|sap|ibm|infosys|tcs|wipro|accenture|salesforce|servicenow|snowflake|databricks|"
    r"5g|edge computing|digital twin|metaverse|machine|mobile (?:app|banking)|internet banking|upi|"
    r"scada|distributed control|dcs|smart meter\w*|advanced metering|grid automation|substation|predictive maintenance|industry 4\.0|smart (?:factory|building|grid)|"
    r"electronic (?:medical|health) record\w*|emr|ehr|hospital information|telemedicine|telehealth|lims|serialization|track and trace|"
    r"proptech|building information|bim|construction technology|"
    r"oss|bss|open ran|network automation|virtuali[sz]ation|"
    r"manufacturing execution|plm|telematics|e-?commerce|omni-?channel|d2c|distributor management|route to market|connected (?:car|vehicle)s?|ev charging|"
    r"seismic|digital oilfield|advanced process control)(?![a-z0-9])", re.I)
TABLE_ONLY_ON_TECH_PAGES = True      # run the slow table detector only on pages that mention technology topics
print("Config loaded.")

Config loaded.


## 3. Parsing — PDF → text paragraphs + tables (with page numbers)
* Removes repeated headers/footers (lines that appear on ≥30 % of pages).
* Glues wrapped lines back into paragraphs and remembers the nearest **heading** (the "section").
* Detects real tables and converts them to Markdown so numbers stay aligned.

In [6]:
@dataclass
class Chunk:
    id: int
    text: str
    page: int
    source: str

    kind: str = "text"          # "text" | "table"
    section: str = ""
    company: str = ""

    @property
    def breadcrumb(self) -> str:
        b = f"[{self.source} | page {self.page}"
        if self.section:
            b += f" | section: {self.section[:80]}"
        return b + "]"

    @property
    def embed_text(self) -> str:      # what gets embedded / BM25-indexed
        return f"{self.breadcrumb}\n{self.text}"


def _norm_line(s: str) -> str:
    s = s.replace("\u00ad", "").replace("\ufb01", "fi").replace("\ufb02", "fl")
    s = s.replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
    return re.sub(r"\s+", " ", s).strip()


def _repeated_lines(pages: list, ratio: float = 0.30) -> set:
    """Header/footer/running-title lines that appear on many pages -> drop them."""
    from collections import Counter
    c = Counter()
    for lines in pages:
        for ln in set(lines):
            if 4 <= len(ln) <= 120:
                c[ln] += 1
    th = max(4, int(len(pages) * ratio))
    return {ln for ln, n in c.items() if n >= th}


def _is_heading(ln: str) -> bool:
    if not (3 < len(ln) < 80) or re.search(r"[.:;,]$", ln):
        return False
    if re.match(r"^[\d\s.,%()-]+$", ln):
        return False
    if ln.isupper():
        return True
    words = ln.split()
    caps = sum(1 for w in words if w[:1].isupper())
    return 1 <= len(words) <= 9 and caps >= max(1, len(words) - 1) and not ln.endswith(".")


def _paragraphs(lines: list, section: str = ""):
    """Join wrapped PDF lines into paragraphs; carry the nearest heading as 'section'.
    Returns (paragraphs, last_section) so the section can continue onto the next page."""
    paras, cur = [], ""

    def flush():
        nonlocal cur
        if cur.strip():
            paras.append({"text": cur.strip(), "section": section})
        cur = ""

    for ln in lines:
        if _is_heading(ln):
            flush()
            section = ln
            continue
        if not cur:
            cur = ln
            continue
        if re.search(r"[.:;!?)\]]$", cur) or re.match(r"^[•▪◦\-–—*]\s?", ln) or re.match(r"^\(?\d{1,2}[.)]\s", ln):
            flush()
            cur = ln
        elif cur.endswith("-") and ln[:1].islower():
            cur = cur[:-1] + ln
        else:
            cur = cur + " " + ln
    flush()
    return paras, section


def _table_to_md(t) -> str:
    try:
        md = t.to_markdown()
        if md and md.strip():
            return md.strip()
    except Exception:
        pass
    try:
        rows = t.extract()
        out = []
        for r in rows or []:
            cells = ["" if c is None else re.sub(r"\s+", " ", str(c)).strip() for c in r]
            if any(cells):
                out.append(" | ".join(cells))
        return "\n".join(out)
    except Exception:
        return ""


def parse_pdf(pdf_path, detect_tables: bool = True, drop_repeats: bool = True):
    try:
        import pymupdf
    except ImportError:            # very old PyMuPDF versions
        import fitz as pymupdf
    src = Path(pdf_path).name
    doc = pymupdf.open(str(pdf_path))
    n_pages = len(doc)
    pages_lines, raw_text = [], []
    for page in doc:
        txt = page.get_text("text") or ""
        lines = [l for l in (_norm_line(x) for x in txt.split("\n")) if l]
        pages_lines.append(lines)
        raw_text.append(" ".join(lines))
    reps = _repeated_lines(pages_lines) if drop_repeats else set()

    company = ""
    if raw_text:
        first = [l for l in pages_lines[0][:6] if l]
        for ln in first:
            if 3 < len(ln) < 80 and not re.search(r"\d{4}", ln) and "report" not in ln.lower():
                company = ln
                break
        if not company and first:
            company = first[0]

    blocks = []
    section = ""
    t0 = time.time()
    for pno, page in enumerate(doc):
        lines = [l for l in pages_lines[pno] if l not in reps]
        paras, section = _paragraphs(lines, section)
        for p in paras:
            if len(p["text"]) < CFG["min_chunk_chars"]:
                continue
            blocks.append(Chunk(id=len(blocks), text=p["text"], page=pno + 1, source=src,
                                kind="text", section=p["section"], company=company))
        if detect_tables and (not TABLE_ONLY_ON_TECH_PAGES or TECH_RE.search(raw_text[pno])):
            try:
                for ti, tb in enumerate(page.find_tables().tables):
                    md = _table_to_md(tb)
                    if md and len(md) > 40:
                        blocks.append(Chunk(id=len(blocks), text="TABLE:\n" + md, page=pno + 1,
                                            source=src, kind="table", section=f"table {ti + 1}",
                                            company=company))
            except Exception:
                pass
        if (pno + 1) % 50 == 0:
            print(f"   ... parsed {pno + 1}/{n_pages} pages ({time.time() - t0:.0f}s)")
    doc.close()
    print(f"[parse] {src}: {n_pages} pages -> {len(blocks)} blocks "
          f"({sum(1 for c in blocks if c.kind == 'table')} tables, {len(reps)} repeated lines dropped)")
    if not blocks:
        print("⚠️  No text found. This PDF may be a scanned image (needs OCR first).")
    return blocks, company

## 4. Chunking — pack paragraphs into ~1500-character chunks
* Tables are never split.
* A chunk never spans two pages, so the **page number in every citation is exact**.
* Very long paragraphs are split with a 250-character overlap.

In [7]:
SEPS = ["\n\n", "\n", "; ", ". ", ", ", " "]


def _recursive_split(text: str, size: int, overlap: int) -> list:
    if len(text) <= size:
        return [text]
    cut, sep = -1, ""
    for s in SEPS:
        i = text.rfind(s, 0, size)
        if i > int(size * 0.45):
            cut, sep = i + len(s), s
            break
    if cut <= 0:
        i = text.rfind(" ", 0, size)
        cut, sep = (i + 1 if i > int(size * 0.45) else size), ""
    head = text[:cut].strip()
    tail = text[max(0, cut - overlap):].strip()
    if len(tail) >= len(text.strip()):
        tail = text[cut:].strip()
    parts = [head] if head else []
    if tail:
        parts += _recursive_split(tail, size, overlap)
    return parts


def chunk_blocks(blocks: list) -> list:
    """Tables stay whole; text blocks are packed paragraph-wise up to chunk_size (never across pages)."""
    out, buf, buf_len = [], [], 0
    size, overlap = CFG["chunk_size"], CFG["chunk_overlap"]

    def flush(keep_overlap: bool = False):
        nonlocal buf, buf_len
        if not buf:
            return
        text = "\n\n".join(c.text for c in buf)
        sec = next((c.section for c in reversed(buf) if c.section), "")
        out.append(Chunk(id=len(out), text=text, page=buf[0].page, source=buf[0].source,
                         kind="text", section=sec, company=buf[0].company))
        last = buf[-1]
        buf, buf_len = [], 0
        if keep_overlap and len(last.text) < overlap:      # repeat a short last paragraph as overlap
            buf, buf_len = [last], len(last.text) + 2

    for b in blocks:
        if b.kind == "table":
            flush()
            out.append(Chunk(id=len(out), text=b.text, page=b.page, source=b.source,
                             kind="table", section=b.section, company=b.company))
            continue
        if len(b.text) >= size * 1.5:                       # very long paragraph
            flush()
            for part in _recursive_split(b.text, size, overlap):
                out.append(Chunk(id=len(out), text=part, page=b.page, source=b.source,
                                 kind="text", section=b.section, company=b.company))
            continue
        if buf and buf[0].page != b.page:                   # new page -> new chunk (exact page citations)
            flush()
        elif buf and buf_len + len(b.text) > size:
            flush(keep_overlap=True)
        buf.append(b)
        buf_len += len(b.text) + 2
    flush()
    for i, c in enumerate(out):
        c.id = i
    if out:
        print(f"[chunk] {len(blocks)} blocks -> {len(out)} chunks "
              f"(avg {int(np.mean([len(c.text) for c in out]))} chars)")
    return out

## 5. Embeddings + reranker — these run on your **GPU**
* `Embedder` uses `sentence-transformers` on CUDA in half precision (fp16) — fast and light on VRAM.
* `Reranker` is a cross-encoder that reads *(question, chunk)* pairs together and scores them more accurately than vector search alone.
* Both fall back to the CPU automatically if no GPU is found.

In [8]:
def _tokenize(t: str) -> list:
    toks = re.findall(r"[a-z0-9][a-z0-9+#.\-]*", t.lower())
    out = []
    for tk in toks:
        tk = tk.strip(".-")
        if not tk or tk in STOP or len(tk) < 2:
            continue
        if len(tk) > 5:
            if tk.endswith("ies"):
                tk = tk[:-3] + "y"
            elif tk.endswith("ing") and len(tk) > 6:
                tk = tk[:-3]
            elif tk.endswith("ed") and len(tk) > 5:
                tk = tk[:-2]
            elif tk.endswith("s") and not tk.endswith("ss"):
                tk = tk[:-1]
        out.append(tk)
    return out


def _l2(m: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(m, axis=-1, keepdims=True)
    n[n == 0] = 1.0
    return m / n


class Embedder:
    """Sentence-transformers embedder (GPU if available)."""
    BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

    def __init__(self, model: str = None, device: str = None, batch_size: int = 16):
        from sentence_transformers import SentenceTransformer
        self.model_name = model or CFG["embed_model"]
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.model = SentenceTransformer(self.model_name, device=self.device)
        self.model.max_seq_length = 384
        if self.device == "cuda":
            self.model.half()                       # fp16: ~2x faster, half the VRAM
        name = self.model_name.lower()
        self.query_prefix = self.BGE_QUERY_PREFIX if ("bge" in name and "-en" in name) else ""
        print(f"[embed] model={self.model_name} device={self.device}")

    def encode_documents(self, texts: Iterable) -> np.ndarray:
        """Batch-wise, length-sorted, with a plain-text progress line (no widgets, low RAM)."""
        texts = list(texts)
        n = len(texts)
        order = np.argsort([len(t) for t in texts])
        out = np.zeros((n, self.model.get_sentence_embedding_dimension()), dtype=np.float32)
        group, t0, next_report = 64, time.time(), 0.1
        for s0 in range(0, n, group):
            ids = order[s0:s0 + group]
            v = self.model.encode([texts[i] for i in ids], batch_size=self.batch_size, normalize_embeddings=True,
                                  show_progress_bar=False, convert_to_numpy=True)
            out[ids] = v
            if (s0 + group) / n >= next_report:
                print(f"   ... embedded {min(s0 + group, n)}/{n} chunks ({time.time() - t0:.0f}s)", flush=True)
                next_report += 0.1
        return _l2(out)

    def encode_query(self, text: str) -> np.ndarray:
        v = self.model.encode([self.query_prefix + text], normalize_embeddings=True,
                              show_progress_bar=False, convert_to_numpy=True)
        return _l2(np.asarray(v, dtype=np.float32))[0]


class Reranker:
    """Cross-encoder reranker (GPU if available). Falls back to fused order if unavailable."""

    def __init__(self, model: str = None, enabled: bool = True, device: str = None):
        self.model, self.ok = None, False
        if not enabled:
            print("[rerank] disabled")
            return
        name = model or CFG["reranker_model"]
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        try:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder(name, device=device, max_length=384)
            if device == "cuda":
                try:
                    self.model.model.half()
                except Exception:
                    pass
            self.ok = True
            print(f"[rerank] model={name} device={device}")
        except Exception as e:
            print(f"[rerank] unavailable ({e}) -> using fused retrieval order")

    def score(self, query: str, docs: list) -> list:
        if not self.ok or not docs:
            return [0.0] * len(docs)
        try:
            s = self.model.predict([(query, d) for d in docs], batch_size=16, show_progress_bar=False)
            return [float(x) for x in s]
        except Exception as e:
            print(f"[rerank] scoring failed ({e}); falling back to fused order")
            self.ok = False
            return [0.0] * len(docs)

## 6. Query expansion
A question like *"What AI initiatives…"* is expanded into several variants containing synonyms (*machine learning, generative AI, copilot…*). Every variant is searched, and the results are merged — this makes retrieval much more robust to wording differences between your question and the report.

In [9]:
def _has_term(text: str, term: str) -> bool:
    """Whole-word match (so 'ai' does not fire inside 'maintain', 'erp' not inside 'enterprise')."""
    return re.search(r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])", text) is not None


def expand_query(question: str, hint_keys: list = None) -> list:
    variants = [question]
    ql = question.lower()
    hit_groups, used = [], set()
    for term, group in SYN_LOOKUP.items():
        if term not in used and _has_term(ql, term):
            hit_groups.append(group)
            used.update(group)
    if hint_keys:
        for k in hint_keys:
            g = SYN_LOOKUP.get(k)
            if g:
                hit_groups.append(g)
                used.update(g)
    for g in hit_groups[:5]:
        extra = " ".join(t for t in g if t not in ql)
        if extra:
            variants.append(f"{question} {extra}")
    if hit_groups:
        kw = []
        for g in hit_groups[:6]:
            kw.extend(g[:6])
        variants.append(question + " " + " ".join(dict.fromkeys(kw)))
    seen, out = set(), []
    for v in variants:
        k = v.lower().strip()
        if k not in seen:
            seen.add(k)
            out.append(v)
    return out


# quick self-test (no PDF or GPU needed)
for _v in expand_query("What AI initiatives has the company undertaken?"):
    print("*", _v[:130])

* What AI initiatives has the company undertaken?
* What AI initiatives has the company undertaken? artificial intelligence machine learning deep learning generative ai genai ai/ml d
* What AI initiatives has the company undertaken? artificial intelligence machine learning deep learning generative ai genai ai/ml


## 7. Hybrid index — dense + BM25 + rank fusion + rerank
`retrieve()` does: expand → dense search + BM25 search for every variant → **Reciprocal Rank Fusion** → cross-encoder rerank → keep top-k → add neighbouring chunks from the same page.

In [10]:
class HybridIndex:
    def __init__(self, chunks: list, embedder: Embedder, reranker: Reranker):
        self.chunks, self.embedder, self.reranker = chunks, embedder, reranker
        self.vectors = None
        self.bm25 = None
        self.tokens = []

    def _build_bm25(self):
        try:
            from rank_bm25 import BM25Okapi
            self.bm25 = BM25Okapi(self.tokens)
        except Exception as e:
            self.bm25 = None
            print(f"[bm25] unavailable ({e}); dense-only retrieval")

    # ---- build ----
    def build(self):
        self.tokens = [_tokenize(c.embed_text) for c in self.chunks]
        self._build_bm25()
        t0 = time.time()
        self.vectors = self.embedder.encode_documents([c.embed_text for c in self.chunks])
        print(f"[index] embedded {len(self.chunks)} chunks in {time.time() - t0:.1f}s shape={self.vectors.shape}")
        return self

    # ---- save / load ----
    def save(self, out_dir):
        out = Path(out_dir)
        out.mkdir(parents=True, exist_ok=True)
        (out / "chunks.json").write_text(json.dumps([asdict(c) for c in self.chunks]), encoding="utf-8")
        np.save(out / "vectors.npy", self.vectors)
        (out / "meta.json").write_text(json.dumps({"chunks": len(self.chunks),
                                                   "embed_model": self.embedder.model_name,
                                                   "config": CFG}, indent=2), encoding="utf-8")
        print(f"[index] saved to {out.resolve()} (chunks.json, vectors.npy, meta.json)")

    @classmethod
    def load(cls, out_dir, with_reranker: bool = True):
        out = Path(out_dir)
        meta = json.loads((out / "meta.json").read_text(encoding="utf-8"))
        chunks = [Chunk(**d) for d in json.loads((out / "chunks.json").read_text(encoding="utf-8"))]
        idx = cls(chunks, Embedder(meta.get("embed_model")), Reranker(enabled=with_reranker))
        idx.vectors = np.load(out / "vectors.npy")
        idx.tokens = [_tokenize(c.embed_text) for c in chunks]
        idx._build_bm25()
        print(f"[index] loaded {len(chunks)} chunks from {out}")
        return idx

    # ---- search ----
    def _dense(self, qvec: np.ndarray, k: int) -> list:
        sims = self.vectors @ qvec
        return list(np.argsort(-sims)[:min(k, len(sims))])

    def _sparse(self, query: str, k: int) -> list:
        if self.bm25 is None:
            return []
        scores = np.asarray(self.bm25.get_scores(_tokenize(query)))
        return list(np.argsort(-scores)[:min(k, len(scores))])

    def retrieve(self, question: str, hint_keys: list = None, top_k: int = None, verbose: bool = False) -> list:
        top_k = top_k or CFG["final_top_k"]
        variants = expand_query(question, hint_keys)
        if verbose:
            print(f"[retrieve] {len(variants)} query variants")
            for v in variants:
                print("   *", v[:160])

        dense_lists, sparse_lists, seen = [], [], set()
        for v in variants:
            d = self._dense(self.embedder.encode_query(v), CFG["dense_top_n"])
            s = self._sparse(v, CFG["bm25_top_n"])
            dense_lists.append(d)
            sparse_lists.append(s)
            seen.update(d)
            seen.update(s)
        if not seen:
            return []

        # Reciprocal Rank Fusion across all (variant x retriever) ranked lists
        rrf = {}
        for lists in (dense_lists, sparse_lists):
            for lst in lists:
                for rank, cid in enumerate(lst):
                    rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (60 + rank + 1)
        pool = sorted(rrf, key=rrf.get, reverse=True)[:CFG["rerank_candidates"]]
        if verbose:
            print(f"[retrieve] fused pool={len(pool)} (rerank={self.reranker.ok})")

        # Cross-encoder rerank against the ORIGINAL question
        scores = self.reranker.score(question, [self.chunks[i].embed_text[:1800] for i in pool])
        order = [pool[i] for i in np.argsort(-np.asarray(scores))] if any(scores) else pool
        top = order[:top_k]

        # neighbour expansion: keep answers from losing the next paragraph of the same page
        if CFG["neighbor_expand"]:
            ids = set(top)
            for i in list(top):
                for j in (i - 1, i + 1):
                    if 0 <= j < len(self.chunks) and self.chunks[j].page == self.chunks[i].page:
                        ids.add(j)
            top = sorted(ids, key=lambda i: (self.chunks[i].page, i))
        picked = [self.chunks[i] for i in top]
        if verbose:
            for c in picked[:12]:
                print(f"   - p.{c.page:<4} {c.kind:<5} {len(c.text):>5} chars | {c.text[:90]!r}")
        return picked

## 8. LLM backends, prompts and the pipeline class
The default is **Ollama on your GPU**. Other options (all free tiers) also work by changing one string in Section 9:

| `LLM = ...` | Meaning |
|---|---|
| `"ollama:qwen3:8b"` | local model on your RTX 4060 (default) |
| `"groq:openai/gpt-oss-120b"` | Groq free tier (set env var `GROQ_API_KEY`) |
| `"gemini:gemini-flash-latest"` | Gemini free tier (set env var `GEMINI_API_KEY`) |
| `"mock:extractive"` | **no LLM at all** – just returns the retrieved sentences with pages (good for testing) |

In [11]:
@dataclass
class LLMSpec:
    provider: str
    model: str
    base_url: str = ""
    api_key: str = ""

    def describe(self) -> str:
        return f"{self.provider}:{self.model}"


def parse_llm_spec(spec: str) -> LLMSpec:
    """ollama:qwen3:8b | groq:openai/gpt-oss-120b | gemini:gemini-flash-latest | mock:extractive
       openai:<model>@<base_url>   (OpenRouter / LM Studio / vLLM / any OpenAI-compatible)"""
    if ":" not in spec:
        raise ValueError("LLM must look like 'ollama:qwen3:8b', 'groq:<model-id>', "
                         "'gemini:<model-id>', 'mock:extractive' or 'openai:<model>@<base-url>'")
    provider, rest = spec.split(":", 1)
    provider = provider.lower()
    if provider == "ollama":
        return LLMSpec("ollama", rest or "qwen3:8b", os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434"))
    if provider == "groq":
        return LLMSpec("groq", rest or "openai/gpt-oss-120b",
                       os.environ.get("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
                       os.environ.get("GROQ_API_KEY", ""))
    if provider == "gemini":
        return LLMSpec("gemini", rest or "gemini-flash-latest", "", os.environ.get("GEMINI_API_KEY", ""))
    if provider == "mock":
        return LLMSpec("mock", rest or "extractive")
    if provider in ("openai", "compat", "openrouter"):
        model, _, base = rest.partition("@")
        return LLMSpec("openai", model.strip(), (base or os.environ.get("LLM_BASE_URL", "")).strip(),
                       os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY", ""))
    raise ValueError(f"Unknown provider '{provider}'")


def _post_json(url: str, payload: dict, headers: dict, timeout: int = 600) -> dict:
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json", "Accept": "application/json",
                                          "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AnnualReportRAG/1.0",
                                          **headers})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode(errors='replace')[:300]}") from None


def _get_json(url: str, headers: dict, timeout: int = 60):
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AnnualReportRAG/1.0", **headers}), timeout=timeout) as r:
        return json.loads(r.read().decode())


def _strip_think(text: str) -> str:
    """Remove <think>...</think> reasoning blocks that some models (qwen3, deepseek-r1) emit."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def mock_extractive_answer(user_prompt: str, max_bullets: int = 12) -> str:
    """No-LLM fallback: returns the best-matching report sentences WITH page citations."""
    ctx = user_prompt.split("CONTEXT (each excerpt is labelled with its page in the report)", 1)[-1]
    ctx = ctx.split("RULES\n-----", 1)[0]
    blocks = [b for b in ctx.split("--- EXCERPT (") if b.strip()]
    out, pages = [], []
    for b in blocks[:max_bullets]:
        head, _, body = b.partition(") ---")
        page = re.search(r"page (\d+)", head)
        pg = page.group(1) if page else "?"
        body = re.sub(r"\s+", " ", body).strip()
        keep, ln = [], 0
        for s in re.split(r"(?<=[.!?])\s+", body):
            if len(s) < 25:
                continue
            keep.append(s)
            ln += len(s)
            if ln > 320:
                break
        if keep:
            pages.append(pg)
            out.append("- " + " ".join(keep) + f" (p.{pg})")
    if not out:
        return "The annual report evidence retrieved for this question does not provide sufficient information."
    return ("_Extractive answer (mock backend, no LLM): the report sentences retrieved for this "
            "question, each with its page._\n\n" + "\n".join(out) +
            f"\n\nEvidence used: {', '.join('p.' + p for p in dict.fromkeys(pages))}")


class RateLimitStop(RuntimeError):
    """Raised when the provider says 'try again in a long time' (e.g. daily token limit reached)."""


def _retry_after_seconds(msg: str):
    m = re.search(r"try again in\s+((?:\d+(?:\.\d+)?(?:ms|h|m|s)\s*)+)", msg)
    if not m:
        return None
    tot = 0.0
    for v, u in re.findall(r"(\d+(?:\.\d+)?)(ms|h|m|s)", m.group(1)):
        v = float(v)
        tot += v / 1000 if u == "ms" else v * 3600 if u == "h" else v * 60 if u == "m" else v
    return tot


def llm_complete(spec: LLMSpec, system: str, user: str, temperature: float = 0.0,
                 max_tokens: int = 3000, retries: int = 8) -> str:
    if spec.provider == "mock":
        return mock_extractive_answer(user)
    last = None
    for attempt in range(retries):
        try:
            if spec.provider == "ollama":
                payload = {"model": spec.model, "stream": False, "keep_alive": "30m",
                           "messages": [{"role": "system", "content": system},
                                        {"role": "user", "content": user}],
                           "options": {"temperature": temperature, "num_ctx": CFG["ollama_num_ctx"],
                                       "num_predict": max_tokens}}
                if any(k in spec.model.lower() for k in ("qwen3", "deepseek-r1")):
                    payload["think"] = False          # skip the slow "thinking" phase
                elif "gpt-oss" in spec.model.lower():
                    payload["think"] = globals().get("LLM_REASONING", "low")   # gpt-oss: reasoning level (cannot be switched off)
                out = _post_json(f"{spec.base_url}/api/chat", payload, {})
                return _strip_think(out["message"]["content"])

            if spec.provider == "gemini":
                if not spec.api_key:
                    raise ValueError("GEMINI_API_KEY is not set")
                url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
                       f"{spec.model}:generateContent?key={spec.api_key}")
                out = _post_json(url, {
                    "systemInstruction": {"parts": [{"text": system}]},
                    "contents": [{"role": "user", "parts": [{"text": user}]}],
                    "generationConfig": {"temperature": temperature, "maxOutputTokens": max_tokens}}, {})
                return out["candidates"][0]["content"]["parts"][0]["text"]

            # OpenAI-compatible (groq, openrouter, lm studio, vllm, ...)
            base = spec.base_url.rstrip("/")
            if not base:
                raise ValueError("base_url required (set LLM_BASE_URL or use model@https://...)")
            if spec.provider == "groq" and not spec.api_key:
                raise ValueError("GROQ_API_KEY is not set")
            body = {
                "model": spec.model,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
                "temperature": temperature, "max_tokens": max_tokens}
            if "gpt-oss" in spec.model.lower():
                body["reasoning_effort"] = globals().get("LLM_REASONING", "low")   # "low" | "medium" | "high" (set LLM_REASONING in Section 9)
            out = _post_json(f"{base}/chat/completions", body,
                {"Authorization": f"Bearer {spec.api_key}"} if spec.api_key else {})
            ch = out["choices"][0]
            txt = (ch["message"].get("content") or "")
            if not txt.strip():
                raise RuntimeError("empty LLM response (reasoning used all tokens?)")
            if ch.get("finish_reason") == "length" and attempt < 2 and max_tokens < 6000:
                raise RuntimeError("truncated LLM response (finish_reason=length)")
            return txt
        except ValueError:
            raise
        except urllib.error.URLError as e:
            if spec.provider == "ollama":
                raise RuntimeError("Cannot reach Ollama at " + spec.base_url + ". Is Ollama running? "
                                   "(install from ollama.com, then run: ollama pull " + spec.model + ")") from None
            last = e
        except Exception as e:
            last = e
            msg = str(e)
            if "empty LLM response" in msg or "truncated LLM response" in msg:
                max_tokens = min(int(max_tokens * 1.6), 6000)      # give the model more room and retry
            if any(c in msg for c in ("HTTP 400", "HTTP 401", "HTTP 403", "HTTP 404")):
                break                                   # not retryable: bad key / blocked / wrong model name
            if "429" not in msg and "rate" not in msg.lower() and attempt >= 2:
                break
        wait = 2 ** attempt
        if last is not None and "429" in str(last):
            ra = _retry_after_seconds(str(last))
            if ra is not None and ra > 150:
                raise RateLimitStop(f"Provider rate/daily limit reached - try again in about {ra / 60:.0f} min. "
                                    f"Finished cells are cached; run Section 12 again later. ({str(last)[:200]})")
            wait = min(90, (ra + 1.5) if ra is not None else 8 * (attempt + 1))
        print(f"[llm] retry {attempt + 1}/{retries} after error: {str(last)[:160]} (sleep {wait}s)")
        time.sleep(wait)
    raise RuntimeError(f"LLM call failed: {last}")


def llm_list_models(spec: LLMSpec) -> list:
    try:
        if spec.provider == "ollama":
            return [m["name"] for m in _get_json(f"{spec.base_url}/api/tags", {}).get("models", [])]
        if spec.provider == "gemini":
            url = f"https://generativelanguage.googleapis.com/v1beta/models?key={spec.api_key}"
            return [m["name"].split("/")[-1] for m in _get_json(url, {}).get("models", [])]
        base = (spec.base_url or "https://api.groq.com/openai/v1").rstrip("/")
        out = _get_json(f"{base}/models", {"Authorization": f"Bearer {spec.api_key}"} if spec.api_key else {})
        return sorted(d.get("id", "?") for d in out.get("data", []))
    except Exception as e:
        return [f"<could not list models: {e}>"]


# ---------------------------------------------------------------- prompts
SYSTEM = (
    "You are a meticulous equity-research analyst who reads company annual reports. "
    "You answer strictly and only from the retrieved report excerpts supplied to you. "
    "You never use outside knowledge, never guess numbers, and never fill gaps with "
    "plausible-sounding corporate language. Every factual sentence you write carries the "
    "page citation of the excerpt it came from, in the form (p.34). If the excerpts do not "
    "contain the answer, you say so explicitly and name what is missing."
)

ANSWER_TEMPLATE = """Answer the QUESTION below using ONLY the CONTEXT excerpts from one company's annual report.

QUESTION
--------
{question}

CONTEXT (each excerpt is labelled with its page in the report)
--------------------------------------------------------------
{context}

RULES
-----
1. Use only the CONTEXT. No outside knowledge, no inference beyond what is written.
2. Cite the page after every factual claim, e.g. "AdaniConneX plans a 2 GW data-centre footprint (p.114)".
   Use the page number printed in the excerpt header, not the excerpt number.
3. If the CONTEXT only partially answers the question, answer the part that is supported and then add a
   short line: "Not disclosed in the retrieved sections:" followed by what is missing.
4. If a figure (capex, R&D spend, investment amount) is asked for and the exact number is not present,
   write "Exact figure not disclosed in the retrieved sections" and give the closest qualitative statement.
5. Prefer the company's own wording for named technologies, platforms and partners. Keep exact names
   (product names, vendor names, programme names) verbatim.
6. Structure the answer with short bold headings when the answer has more than one theme.
   Be dense and specific - no filler, no repetition of the question, no generic commentary.
7. End with a line "Evidence used: " listing the distinct page numbers you relied on.

ANSWER
------"""


def build_context(chunks: list, budget: int = None) -> str:
    budget = budget or CFG["max_context_chars"]
    parts, used, seen_text = [], 0, set()
    for c in chunks:
        key = c.text[:200]
        if key in seen_text:
            continue
        seen_text.add(key)
        block = (f"--- EXCERPT ({c.source}, page {c.page}{', TABLE' if c.kind == 'table' else ''}"
                 f"{', section: ' + c.section[:70] if c.section else ''}) ---\n{c.text}")
        if used + len(block) > budget:
            continue
        parts.append(block)
        used += len(block)
    return "\n\n".join(parts)


# ---------------------------------------------------------------- pipeline
class AnnualReportRAG:
    def __init__(self, index: HybridIndex, llm: LLMSpec = None):
        self.index, self.llm = index, llm

    @classmethod
    def build(cls, pdf_path, out_dir=None, llm: str = None, use_reranker: bool = True, detect_tables: bool = True):
        if not Path(pdf_path).exists():
            raise FileNotFoundError(f"PDF not found: {Path(pdf_path).resolve()}  -> fix PDF_PATH in Section 9")
        blocks, _ = parse_pdf(pdf_path, detect_tables=detect_tables)
        chunks = chunk_blocks(blocks)
        if not chunks:
            raise RuntimeError("No text could be extracted from the PDF (scanned image?).")
        idx = HybridIndex(chunks, Embedder(), Reranker(enabled=use_reranker)).build()
        if out_dir:
            idx.save(out_dir)
        return cls(idx, parse_llm_spec(llm) if llm else None)

    @classmethod
    def load(cls, index_dir, llm: str = None, use_reranker: bool = True):
        return cls(HybridIndex.load(index_dir, with_reranker=use_reranker), parse_llm_spec(llm) if llm else None)

    def answer(self, question: str, hint_keys: list = None, top_k: int = None, verbose: bool = False) -> dict:
        chunks = self.index.retrieve(question, hint_keys=hint_keys, top_k=top_k, verbose=verbose)
        context = build_context(chunks)
        result = {"question": question, "context_chars": len(context),
                  "sources": [{"page": c.page, "kind": c.kind, "section": c.section, "preview": c.text[:160]}
                              for c in chunks]}
        if not self.llm:
            result["answer"] = None
            return result
        t0 = time.time()
        result["answer"] = llm_complete(self.llm, SYSTEM, ANSWER_TEMPLATE.format(question=question, context=context))
        result["llm_seconds"] = round(time.time() - t0, 1)
        return result

    def answer_suite(self, questions: list = None, sleep_between: float = 0.0, out_md=None) -> str:
        questions = questions or THE_14_QUESTIONS
        md_parts = [f"# Technology due-diligence Q&A - {self.index.chunks[0].source}\n",
                    f"_Model: {self.llm.describe() if self.llm else 'retrieval only'}_\n"]
        for i, q in enumerate(questions, 1):
            print(f"Q{i}/{len(questions)}: {q}")
            r = self.answer(q, hint_keys=QUESTION_HINTS.get(i) if questions is THE_14_QUESTIONS else None)
            pages = sorted({s["page"] for s in r["sources"]})
            if r["answer"] is None:
                md_parts.append(f"## Q{i}. {q}\n\n`retrieval only`\n\n"
                                + "\n".join(f"- p.{s['page']} [{s['kind']}] {s['preview']}" for s in r["sources"]))
            else:
                md_parts.append(f"## Q{i}. {q}\n\n{r['answer']}\n\n<sub>retrieved pages: {pages}</sub>\n")
                display(Markdown(f"### Q{i}. {q}\n\n{r['answer']}\n\n<sub>retrieved pages: {pages}</sub>"))
            if sleep_between:
                time.sleep(sleep_between)
        text = "\n".join(md_parts)
        if out_md:
            Path(out_md).write_text(text, encoding="utf-8")
            print(f"\n[suite] report written to {Path(out_md).resolve()}")
        return text


def show(result: dict):
    """Pretty-print one answer in the notebook."""
    body = result["answer"] or "_(retrieval only - no LLM configured)_"
    pages = sorted({s["page"] for s in result["sources"]})
    display(Markdown(f"### {result['question']}\n\n{body}\n\n<sub>retrieved pages: {pages}</sub>"))

print("Pipeline ready.")

Pipeline ready.


## 9. ▶️ RUN IT — edit only this cell
Set `START_COMPANY` / `END_COMPANY` (e.g. 1 and 2), pick the `LLM`, then run Sections 9 → 13 in order.
Change the range and run again from Section 9: new companies are **appended** to the same Excel (`tech_matrix_all.xlsx`); companies already in it are skipped.

In [12]:
# ================== EDIT ONLY THIS CELL BEFORE EACH RUN ==================
REPORTS_DIR   = "./annual_reports"        # folder that contains ALL company annual-report PDFs
COMPANY_CSV   = "./Company_data.csv"      # your company list (.csv or .xlsx) - names in the first column
OUTPUT_DIR    = "./tech_matrix_output"    # cache, indexes, match report and the final Excel go here
EXCEL_FILE    = "tech_matrixs_all.xlsx"    # ONE Excel for all companies (inside OUTPUT_DIR)

# ---- which companies to run (position in the company list, 1-based, both ends included) ----
START_COMPANY = 1
END_COMPANY   = 359

# ---- LLM (runs in the cloud, so your 8 GB laptop is fine) ----
LLM = "groq:openai/gpt-oss-120b"      # needs a free Groq key (Section 9b)
# LLM = "groq:openai/gpt-oss-20b"     # smaller/faster gpt-oss on Groq - try if you hit token limits
# LLM = "gemini:gemini-flash-latest"  # alternative free cloud option (env var GEMINI_API_KEY)
# LLM = "mock:extractive"             # no LLM - only for testing the flow
# (Do NOT use ollama:gpt-oss:20b - it needs ~16 GB RAM/VRAM; this laptop has 8 GB.)

LLM_REASONING  = "medium"   # gpt-oss thinking level: "low" (fastest) | "medium" | "high"
USE_RERANKER    = True
EMBED_ONLY_TECH_CHUNKS = False   # True = embed only chunks that mention technology topics (3-5x faster, far less RAM)
TOP_K_PER_QUERY = 10        # chunks retrieved per query (raise to 8-10 if many cells say "Not disclosed")
CONTEXT_CHARS   = 14000    # max report text sent to the LLM per parameter (smaller = fewer Groq tokens)
MAX_BULLETS     = 20      # max bullets per Excel cell
LLM_PAUSE       = 6.0      # seconds to wait between LLM calls (helps with free-tier per-minute limits)

MATCH_THRESHOLD = 0.80     # name-match score needed to accept a PDF automatically (0-1)
MANUAL_PDF_MAP  = {        # force a company -> PDF file when auto-match fails (see match_report.csv)
    # "Tata Consultancy Services Ltd.": "TCS_AR_2025.pdf",
}
COMPANY_NAMES = {}         # (kept for compatibility with the helper functions)

REBUILD_INDEX = False      # True = re-parse + re-embed the PDFs
FORCE_RERUN   = False     # True = ignore cache AND overwrite companies already in the Excel

### 9b. Groq API key
Run the cell below once (skip it if you use Ollama or `mock`). Paste your `gsk_...` key into the box that appears.

In [13]:

# ---- GROQ API KEY ----
import os
import getpass
from pathlib import Path

key_file = Path("groq_key.txt")

# Load the key from the actual key file
if key_file.exists():
    os.environ["GROQ_API_KEY"] = key_file.read_text(
        encoding="utf-8"
    ).strip()

# Ask for the key only if the file is missing or empty
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass(
        "Paste your Groq API key (gsk_...): "
    )

print("Groq key loaded:", bool(os.environ.get("GROQ_API_KEY")))


Groq key loaded: True


## 10. Table columns (themes)
Each theme = one column. Each has a **focus** (what the LLM must capture: initiatives, investments, partners, amounts), one or more **retrieval queries**, and **hints** (synonym groups used for query expansion).
Add / remove / rename themes here freely — the table columns follow this dictionary, in this order.

In [14]:
# --- extra synonym groups for query expansion --------------------------------------------------------------
SYNONYMS.setdefault("digital", ["digital transformation", "digitalisation", "digitalization", "digital platform",
    "digital initiative", "process automation", "digital channel", "omnichannel", "mobile app", "digital first",
    "technology modernisation", "legacy modernization", "it transformation"])
SYNONYMS.setdefault("data", ["data analytics", "analytics", "big data", "data lake", "data warehouse", "data platform",
    "data governance", "business intelligence", "power bi", "snowflake", "databricks", "data engineering", "data management"])
SYNONYMS.setdefault("automation", ["automation", "rpa", "robotic process automation", "intelligent automation",
    "straight through processing", "stp", "chatbot", "workflow automation", "low code", "no code", "bots"])
SYNONYMS.setdefault("roadmap", ["roadmap", "future technology", "next phase", "going forward", "planned", "pipeline",
    "will invest", "vision", "strategic priorities", "technology strategy", "next generation", "outlook"])
SYNONYMS.setdefault("pain", ["legacy system", "legacy systems", "technical debt", "integration challenge", "skill gap",
    "talent shortage", "data quality", "silos", "cost pressure", "adoption challenge", "modernisation challenge",
    "key audit matter", "it general controls"])
SYNONYMS.setdefault("techabs", ["technology absorption", "technology adaptation", "imported technology",
    "expenditure on research and development", "benefits derived", "innovation efforts", "indigenisation"])
SYNONYMS.setdefault("intangible", ["intangible assets", "computer software", "software licences", "capital work in progress",
    "capitalised software", "it infrastructure", "information technology", "computers and hardware", "data centre"])
SYNONYMS.setdefault("ot", ["scada", "dcs", "distributed control system", "plc", "industrial control", "ot security",
    "predictive maintenance", "digital twin", "industry 4.0", "iiot", "smart factory", "manufacturing execution system"])
for _head, _terms in SYNONYMS.items():
    for _t in _terms:
        SYN_LOOKUP.setdefault(_t, _terms)
    SYN_LOOKUP.setdefault(_head, _terms)

COMMON = ("For each item give: WHAT the initiative/system/programme is, any NAMED partner, vendor, alliance or "
          "subsidiary involved, and any INVESTMENT / spend / metric (with currency and unit) if stated.")

# ----- THE 15 PARAMETERS (Excel columns follow this order). Each has 3-4 queries aimed at DIFFERENT parts of the report -----
THEMES = {
    "Current Technology Adoption": dict(
        focus="Technology and digital systems the company ALREADY uses today in operations and in customer/partner processes: "
              "enterprise and core systems, digital channels and apps, analytics, automation, connected devices - with adoption "
              "levels, usage numbers or % digital metrics. " + COMMON,
        queries=["What technology systems, digital channels, applications and analytics tools does the company currently use in its operations, and how widely are they adopted?",
                 "digital adoption statistics: percentage of transactions, orders, customers or processes done digitally; number of users of the company's mobile app, portal or online platform",
                 "technology and digitisation initiatives implemented during the year across operations, manufacturing, supply chain, customer service and back office"],
        hints=["platform", "digital", "data"]),
    "Digital Transformation": dict(
        focus="Digital transformation programmes: digital platforms and channels, modernisation of legacy systems, process "
              "re-engineering, digital-first strategy, how technology strategy supports the business goals, new digital products. " + COMMON,
        queries=["What digital transformation programmes, digital platforms and modernisation initiatives does the company describe, and how does technology strategy support its business goals?",
                 "management discussion on technology and digital strategy: transformation journey, digital-first approach, legacy system modernisation, process re-engineering",
                 "new digital platforms, apps, portals and customer or partner facing digital products launched or scaled during the year"],
        hints=["digital", "platform", "investment"]),
    "R&D": dict(
        focus="Research & development and in-house technology development: R&D expenditure (capital, recurring, total, % of turnover), "
              "technology absorption/adaptation/innovation efforts, R&D centres and labs, patents, new products, innovation programmes, "
              "research collaborations. " + COMMON,
        queries=["technology absorption adaptation and innovation: expenditure incurred on research and development, capital, recurring, total R&D expenditure as a percentage of total turnover",
                 "R&D centres, innovation labs, patents filed or granted, new products developed and research collaborations with universities or institutes",
                 "innovation programmes, incubators, intellectual property and in-house technology development efforts"],
        hints=["rnd", "techabs", "emerging"]),
    "Emerging Technologies": dict(
        focus="Emerging technologies explored, piloted or adopted: blockchain, IoT, digital twin, quantum, edge, 5G, robotics, drones, "
              "AR/VR, generative AI, Industry 4.0, proofs-of-concept and "
              "-up collaborations. " + COMMON,
        queries=["Which emerging technologies such as blockchain, IoT, digital twin, quantum, edge computing, 5G, robotics, drones, AR/VR or generative AI are being piloted, explored or adopted?",
                 "proof of concept pilots with start-ups, innovation partnerships and new technology experiments",
                 "Industry 4.0, smart factory, connected devices and advanced analytics initiatives"],
        hints=["emerging", "ot"]),
    "Technology Partnerships": dict(
        focus="Named technology partners, vendors, system integrators, hyperscalers, alliances, fintech/start-up tie-ups, MoUs, joint ventures "
              "and research collaborations - with the technology area each relates to (AI, cloud, cyber, data, ERP, network, etc.). " + COMMON,
        queries=["Who are the named technology partners, vendors, system integrators, hyperscalers or alliances (such as Microsoft, AWS, Google, SAP, Oracle, IBM, Infosys, TCS) working with the company?",
                 "memorandum of understanding, strategic collaboration, joint venture or tie-up with technology companies, fintechs, start-ups or research institutions",
                 "outsourcing of IT services, managed service providers, third party technology vendors and vendor management"],
        hints=["vendor", "ai", "cloud", "cyber"]),
    "Technology Investments": dict(
        focus="Money committed to technology: technology/IT capex, investment in software, hardware, data centres, AI, cloud, cyber and digital "
              "platforms, additions to intangible assets (computer software), technology-related acquisitions, JVs, equity or venture investments, "
              "planned technology budgets. Always capture the amount, year and counterparty. " + COMMON,
        queries=["capital expenditure and investment in technology, IT infrastructure, software, digital platforms, data centres, automation and cybersecurity with amounts",
                 "additions to intangible assets computer software licences capital work in progress information technology hardware",
                 "acquisitions, joint ventures, equity investments or venture investments in technology, digital or start-up businesses",
                 "budget or outlay planned for technology and digital initiatives in the coming year"],
        hints=["investment", "capex", "acquisition", "intangible"]),
    "Cloud Adoption": dict(
        focus="Cloud and data infrastructure: cloud migration, public/private/hybrid cloud, cloud-native applications, hyperscaler vendors, "
              "share of workloads on cloud, data centres, disaster-recovery sites, data platforms/lakes/warehouses and related spend. " + COMMON,
        queries=["cloud adoption and migration: public, private or hybrid cloud, cloud-native applications, cloud vendors and the share of workloads on cloud",
                 "data centre, disaster recovery site, business continuity, infrastructure modernisation and hosting arrangements",
                 "data platforms, data lake, data warehouse, data governance and analytics infrastructure"],
        hints=["cloud", "data", "vendor", "intangible"]),
    "Automation": dict(
        focus="Automation: RPA/bots, workflow automation, straight-through processing, chatbots, plant/warehouse/operations automation "
              "(robotics, SCADA, DCS, process control), predictive maintenance, % processes automated, efficiency gains, vendors. " + COMMON,
        queries=["automation initiatives: robotic process automation, bots, workflow automation, straight-through processing, chatbots and results such as hours or cost saved",
                 "factory, plant, warehouse or operations automation: robotics, SCADA, DCS, process control, predictive maintenance and automated production lines",
                 "intelligent automation using AI and machine learning to reduce manual effort, turnaround time or errors"],
        hints=["automation", "digital", "ot"]),
    "Cybersecurity": dict(
        focus="Cybersecurity and information/data security: programmes, security operations centre, certifications (ISO 27001, SOC 2), "
              "zero trust, DLP, VAPT, incident response, governance (board/committee, CISO), data-privacy compliance, security investments, "
              "security vendors and any breaches or privacy complaints reported. " + COMMON,
        queries=["cybersecurity programmes, security operations centre, certifications such as ISO 27001, zero trust, data loss prevention, VAPT and incident response",
                 "information security and cyber risk governance: board or committee oversight, CISO, IT strategy committee, policies and audits",
                 "data privacy and data protection: DPDP Act compliance, customer data privacy complaints, cyber security breaches reported",
                 "cyber insurance, cyber resilience, red team exercises, security awareness training and security vendors"],
        hints=["cyber", "vendor", "investment"]),
    "Future Technology Roadmaps": dict(
        focus="Forward-looking technology plans: technology roadmap, planned platforms and investments, innovation pipeline, targets and "
              "timelines, how the company intends to use AI/digital/automation going forward. " + COMMON,
        queries=["technology roadmap and future plans: planned technology investments, upcoming platforms, targets and timelines for the next financial year",
                 "outlook and strategic priorities: how the company intends to use AI, digital, automation and new technology going forward",
                 "innovation pipeline, upcoming launches, capacity or platform build-out that depends on technology"],
        hints=["roadmap", "emerging", "investment"]),
    "Technology Pain Areas": dict(
        focus="Technology-related pain points and challenges the company itself mentions or that auditors flag: legacy systems, integration "
              "difficulty, IT general controls, data quality, skill shortage, cost of technology, slow adoption, vendor dependency - and the steps taken to address them.",
        queries=["technology related challenges: legacy systems, integration difficulties, data quality, skill shortage, cost of technology, slow adoption, vendor dependency",
                 "key audit matter information technology general controls IT systems and automated controls",
                 "challenges and concerns in management discussion related to digital transformation, cyber threats, talent availability or supply chain visibility"],
        hints=["pain", "risk", "talent"]),
    "AI": dict(
        focus="Artificial intelligence, machine learning, generative AI, copilots, LLMs, computer vision, predictive models: initiatives, "
              "products, use cases, platforms, AI governance/model-risk frameworks, AI centres of excellence, AI investments, partners and benefits achieved. " + COMMON,
        queries=["artificial intelligence, machine learning and generative AI initiatives, use cases, models, copilots and AI platforms deployed by the company",
                 "AI governance, responsible AI, model risk management framework and AI ethics policy",
                 "AI partnerships, AI centre of excellence, AI investments, data science teams and results or benefits achieved from AI"],
        hints=["ai", "vendor"]),
    "Tech Risk": dict(
        focus="Technology-related RISKS disclosed in risk-management / principal-risk sections: cyber threats, data breach/privacy, IT system "
              "failure or outage, obsolescence, third-party/vendor dependency, AI risks, regulatory/IT-compliance risk - and the mitigations described.",
        queries=["risk management: technology risk, cyber security risk, information technology risk, data breach, system failure or outage and mitigation measures",
                 "third party and vendor dependency, outsourcing risk, technology obsolescence, business continuity and disaster recovery",
                 "regulatory and compliance risk related to technology, data privacy, IT governance and AI",
                 "principal risks and uncertainties table with risk description and mitigation plan"],
        hints=["risk", "cyber"]),
    "Platforms & Systems": dict(
        focus="Key platforms, systems and tools in use: ERP (SAP/Oracle), CRM, core/industry systems, proprietary products, data platforms, "
              "operational technology, overall IT landscape and architecture. " + COMMON,
        queries=["key technology platforms, enterprise systems and tools in use such as ERP, SAP, Oracle, CRM, Salesforce, core systems, data platforms and proprietary software",
                 "technology architecture, IT landscape, applications, integration layer, APIs and infrastructure",
                 "sector-specific operational technology and platforms in use in plants, networks, stores, hospitals, projects or branches"],
        hints=["platform", "data", "vendor"]),
    "Talent & Skills": dict(
        focus="Workforce/talent programmes that support technology adoption: upskilling in digital/AI/data/cyber, training hours and numbers "
              "trained, technology hiring, size of technology workforce, academies, certifications, learning partners.",
        queries=["training and upskilling of employees in digital, technology, AI, data and cybersecurity skills, number of employees trained, training hours",
                 "talent acquisition for technology roles, technology workforce size, learning academies, certifications and partnerships with learning providers",
                 "employee training and skill upgradation programmes, percentage of employees trained, BRSR training hours"],
        hints=["talent", "ai"]),
}
assert len(THEMES) == 15, len(THEMES)

# ----- what must NOT be put in each cell (keeps CSR / unrelated items out; fixes noisy cells) -----
AVOID = {
 "Current Technology Adoption": "plans not yet deployed; pure risk-policy statements; CSR or social programmes",
 "Digital Transformation": "CSR/ESG programmes; generic customer-service statements that name no digital platform or technology",
 "R&D": "CSR, foundation or donation spending (that is NOT R&D); sales figures; employee training",
 "Emerging Technologies": "mature everyday systems (standard ERP, ordinary apps); CSR",
 "Technology Partnerships": "regulators and government bodies unless the item is a technology collaboration; customers; CSR beneficiaries",
 "Technology Investments": "CSR spending; capex with no technology link (buildings, land, plant); financial/treasury investments such as securities, loans or an investment portfolio",
 "Cloud Adoption": "generic AI or cybersecurity items unless they specifically concern cloud, data centres or data platforms",
 "Automation": "chatbots or AI statements that do not automate a process; CSR",
 "Cybersecurity": "ESG/financial certifications or opinions unrelated to information security; CSR; physical security",
 "Future Technology Roadmaps": "achievements already completed (they belong in other parameters); generic financial guidance",
 "Technology Pain Areas": "marketing claims and successes; non-technology risks",
 "AI": "non-AI automation; digital statements that do not mention AI/ML",
 "Tech Risk": "non-technology risks (credit, market, liquidity, climate); generic governance statements",
 "Platforms & Systems": "initiatives that name no system or platform; risk or security policies",
 "Talent & Skills": "general HR statistics (hiring numbers, gender mix, well-being) unless tied to technology/digital/AI skills; community skilling or CSR",
}
for _k, _v in AVOID.items():
    THEMES[_k]["avoid"] = _v

# ----- SECTOR PROFILES: what "technology" means in each sector (used for extra retrieval queries AND inside the LLM prompt) -----
SECTOR_QUERY_THEMES = {"Current Technology Adoption", "Digital Transformation", "Emerging Technologies", "Technology Partnerships",
                       "Technology Investments", "Cloud Adoption", "Automation", "Future Technology Roadmaps",
                       "Technology Pain Areas", "AI", "Platforms & Systems"}

SECTOR_DEFAULT = dict(name="the company's", terms="digital platforms ERP automation analytics IoT cloud cybersecurity",
    lens="enterprise systems (ERP/CRM), digital channels, automation, analytics/AI, IoT, cloud and cybersecurity used in operations and customer processes",
    rnd="Capture R&D/innovation spend and technology-absorption disclosures if present.",
    examples="ERP rollout, plant automation, analytics, customer app")

SECTOR_PROFILES = {
  "financialservices": dict(name="Financial Services",
    terms="core banking UPI mobile banking internet banking API banking digital lending video KYC fraud detection AML data centre disaster recovery RBI IT governance fintech partnership digital onboarding",
    lens="core banking / insurance / lending and wealth platforms, mobile and internet banking, UPI and payments, digital lending and onboarding (video-KYC), fraud and AML analytics, API banking and fintech tie-ups, data centres and disaster recovery, regulator-driven IT governance and cyber-security (RBI, IRDAI, SEBI, NHB, CERT-In)",
    rnd="Formal R&D spend is rarely reported; capture innovation labs, fintech incubation, technology/IT expenditure and software/intangible capex instead.",
    examples="UPI volumes, core banking upgrade, video KYC, fraud analytics, API banking, DR site"),
  "healthcare": dict(name="Healthcare",
    terms="hospital information system electronic medical records telemedicine LIMS laboratory automation drug discovery AI data integrity serialization track and trace digital therapeutics connected devices patient app",
    lens="hospital information systems and electronic medical records, telemedicine and patient apps, diagnostics/LIMS automation, AI in drug discovery and imaging, manufacturing automation, data integrity and regulatory IT (USFDA 21 CFR Part 11), serialization/track-and-trace, digital supply chain, cyber security of patient data",
    rnd="R&D is a core disclosure: capture R&D spend (amount, % of sales), R&D centres, pipeline/innovation platforms and digital/AI use in R&D.",
    examples="EMR rollout, AI in drug discovery, LIMS, track and trace, telehealth app, process analytical technology"),
  "technology": dict(name="Technology",
    terms="GenAI platform AI-led services partnerships hyperscaler delivery automation proprietary platform IP patents cloud migration offerings centre of excellence reskilling talent",
    lens="the company's own technology offerings and proprietary platforms/IP, generative-AI and cloud service lines, hyperscaler and ecosystem partnerships, internal delivery automation and AI productivity tools, innovation labs and patents, security services, workforce reskilling at scale",
    rnd="Capture innovation/R&D spend, labs, patents, proprietary platforms and AI/cloud offerings, partners and talent upskilled.",
    examples="GenAI offering, partnership with Microsoft/AWS/Google, innovation lab, patents, reskilled employees"),
  "realestate": dict(name="Real Estate",
    terms="proptech building information modelling BIM smart building IoT CRM digital sales customer app construction technology ERP virtual site visit green building automation",
    lens="construction technology (BIM, precast/aluminium formwork, drones, project-management software), smart buildings and IoT, customer CRM and digital sales / apps, ERP and finance systems, facility-management and building-automation systems, cyber security",
    rnd="R&D is rarely reported; capture construction-technology and innovation initiatives and technology expenditure instead.",
    examples="BIM, CRM/customer app, drone survey, smart-building IoT, ERP"),
  "utilities": dict(name="Utilities",
    terms="SCADA smart meter advanced metering AMI grid automation ADMS digital substation predictive maintenance OT cyber security drone inspection renewable forecasting energy trading platform",
    lens="SCADA and grid automation, smart/advanced metering, ADMS/EMS and digital substations, plant DCS and predictive maintenance, renewable-energy forecasting and trading platforms, drone/robotic inspection, OT cyber security, customer billing/CRM platforms",
    rnd="Capture R&D/technology-absorption disclosures and pilots such as green hydrogen, storage or digital plant projects.",
    examples="smart meters, SCADA, predictive maintenance, ADMS, drone inspection, energy trading platform"),
  "energy": dict(name="Energy",
    terms="digital oilfield seismic AI refinery advanced process control DCS digital twin predictive maintenance OT security trading platform EV charging hydrogen retail digital",
    lens="upstream seismic/reservoir AI and digital oilfield, refinery and petrochemical advanced process control and DCS, digital twins and predictive maintenance, pipeline SCADA and leak detection, trading and risk platforms, retail/EV-charging digital platforms, OT cyber security",
    rnd="Capture R&D spend, R&D centres, patents and technology-absorption disclosures (catalysts, new energy, hydrogen, battery).",
    examples="digital oilfield, APC, digital twin, pipeline SCADA, EV charging platform, R&D centre"),
  "basicmaterials": dict(name="Basic Materials",
    terms="Industry 4.0 DCS PLC advanced process control predictive maintenance digital twin mine automation drones SAP ERP AI process optimisation OT security supply chain",
    lens="plant automation (DCS/PLC, advanced process control), predictive maintenance and digital twins, AI for process and energy optimisation, mine/automation and drones, ERP (SAP) and supply-chain systems, OT cyber security, R&D in products and processes",
    rnd="Capture R&D spend, technology-absorption disclosures, process-innovation and product-development efforts.",
    examples="advanced process control, digital twin of plant, predictive maintenance, drone survey, SAP S/4HANA"),
  "communicationservices": dict(name="Communication Services",
    terms="5G network automation OSS BSS cloud-native core Open RAN AI network optimisation OTT platform data centre edge computing digital customer app cyber security",
    lens="5G/4G network roll-out and virtualisation (cloud-native core, Open RAN), network automation and AI optimisation, OSS/BSS and billing platforms, digital customer apps and OTT/content platforms, data centres and edge, enterprise cloud/IoT services, network cyber security",
    rnd="Capture R&D/innovation spend, in-house 5G/network technology and platforms, patents and technology partnerships.",
    examples="5G roll-out, Open RAN, network AI, OSS/BSS, data centre, OTT platform"),
  "consumercyclical": dict(name="Consumer Cyclical",
    terms="omnichannel e-commerce D2C app CRM loyalty supply chain connected vehicle telematics EV technology factory automation AI personalisation ERP",
    lens="omnichannel/e-commerce and D2C apps, CRM and loyalty analytics, AI personalisation and demand forecasting, supply-chain and inventory systems, factory automation, connected-vehicle/EV technology and telematics (autos), design and product technology, ERP",
    rnd="Capture R&D/product-development spend, design/engineering centres, patents and EV/new-technology programmes.",
    examples="omnichannel platform, D2C app, connected vehicles, EV platform, AI demand forecasting"),
  "consumerdefensive": dict(name="Consumer Defensive",
    terms="distributor management system route to market retailer app demand forecasting IoT cold chain smart manufacturing D2C SAP S/4HANA analytics supply chain visibility",
    lens="distributor and route-to-market systems, retailer/B2B apps, demand forecasting and analytics, supply-chain visibility and cold-chain IoT, smart manufacturing and automation, D2C/e-commerce, ERP (SAP S/4HANA), consumer-insight AI",
    rnd="Capture R&D/innovation spend, R&D centres, new-product pipeline and technology-absorption disclosures.",
    examples="distributor management system, retailer app, demand forecasting, smart factory, S/4HANA"),
  "industrials": dict(name="Industrials",
    terms="Industry 4.0 digital twin IoT telematics PLM MES ERP robotics BIM project management drones predictive maintenance OT cyber security automation engineering design",
    lens="Industry 4.0 and smart-factory systems (MES/PLM/ERP), digital twins and IoT/telematics, robotics and automation in manufacturing and construction, BIM and project-management platforms, drones, predictive maintenance, engineering/design software and AI, OT cyber security",
    rnd="Capture R&D/engineering spend, design and innovation centres, patents and technology-absorption disclosures.",
    examples="digital twin, MES/PLM, IoT telematics, robotics, BIM, predictive maintenance"),
}
print(f"{len(THEMES)} parameters:", " | ".join(THEMES))
print(f"{len(SECTOR_PROFILES)} sector profiles:", " | ".join(p['name'] for p in SECTOR_PROFILES.values()))

15 parameters: Current Technology Adoption | Digital Transformation | R&D | Emerging Technologies | Technology Partnerships | Technology Investments | Cloud Adoption | Automation | Cybersecurity | Future Technology Roadmaps | Technology Pain Areas | AI | Tech Risk | Platforms & Systems | Talent & Skills
11 sector profiles: Financial Services | Healthcare | Technology | Real Estate | Utilities | Energy | Basic Materials | Communication Services | Consumer Cyclical | Consumer Defensive | Industrials


## 11. Helper functions
* **Per-PDF index** — parsed/embedded once (GPU) and cached on disk; the embedding + reranker models are loaded only once for all PDFs.
* **Per-cell extraction** — retrieve the best excerpts for the theme → the LLM writes short bullets → every bullet must end with `(p.N)`.
* **Page check** — any cited page that was *not* in the excerpts given to the LLM is flagged `⚠` (guards against invented page numbers); bullets with no page are flagged too.

> **Page numbers are PDF page numbers** (page 1 = first page of the PDF file), which is what your PDF viewer shows in its page box. They can differ from the page number printed on the report's own page footer (cover / front-matter offset).

In [15]:
CELL_SYSTEM = (
    "You are a meticulous equity-research analyst extracting facts from company annual reports into a comparison table. "
    "You use ON"
    "LY the report excerpts supplied. You never use outside knowledge, never guess, and never invent page numbers. "
    "Every bullet ends with the page citation of the excerpt it came from, like (p.34)."
)

CELL_TEMPLATE = """Fill ONE cell of a comparison table for the company "{company}".
Column: {theme}
What to capture: {focus}

CONTEXT (each excerpt is labelled with its page in the report)
--------------------------------------------------------------
{context}

RULES
-----
1. Use only the CONTEXT. No outside knowledge, no inference beyond what is written.
2. Output ONLY bullet lines, each starting with "- ". Maximum {max_bullets} bullets, most important first. No headings, no intro, no closing remarks.
3. Each bullet: one dense sentence (max ~35 words) with the concrete fact, and it MUST end with the page in brackets, e.g. "(p.34)".
   Use the page number from the excerpt header. If a fact comes from two pages write "(p.34, p.36)".
4. Keep exact names of technologies, platforms, vendors, partners, programmes and subsidiaries verbatim.
   Keep amounts exactly as written with currency and unit (e.g. "Rs 1,200 crore", "USD 45 million"). Never invent a number.
5. Skip generic marketing language that has no concrete initiative, name, number or partner.
6. If the CONTEXT has nothing relevant, output exactly: Not disclosed
"""

CITE_RE = re.compile(r"\((?:[^()]*?\bpp?\.?\s*\d+[^()]*)\)", re.I)


def company_from_filename(p: Path) -> str:
    if p.name in COMPANY_NAMES:
        return COMPANY_NAMES[p.name]
    s = re.sub(r"[_\-]+", " ", p.stem)
    s = re.sub(r"(?i)\bfy\s?\d{2,4}(\s?\d{2,4})?\b", " ", s)
    s = re.sub(r"(?i)\b(integrated|annual|report|reports|ar|fy|final|web|print)\b", " ", s)
    s = re.sub(r"\b(19|20)?\d{2}(\s?\d{2,4})?\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        s = p.stem
    return s.title() if s.islower() else s


def get_index(pdf: Path, company: str, embedder, reranker) -> HybridIndex:
    d = Path(OUTPUT_DIR) / "indexes" / re.sub(r"[^A-Za-z0-9_.-]+", "_", pdf.stem)
    want_model, want_tech = embedder.model_name, bool(EMBED_ONLY_TECH_CHUNKS)
    if (d / "meta.json").exists() and not REBUILD_INDEX:
        meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
        if meta.get("embed_model") == want_model and bool(meta.get("tech_only", False)) == want_tech:
            chunks = [Chunk(**x) for x in json.loads((d / "chunks.json").read_text(encoding="utf-8"))]
            idx = HybridIndex(chunks, embedder, reranker)
            idx.vectors = np.load(d / "vectors.npy")
            idx.tokens = [_tokenize(c.embed_text) for c in chunks]
            idx._build_bm25()
            print(f"[index] loaded cached index for {pdf.name} ({len(chunks)} chunks)")
            return idx
        print(f"[index] cached index for {pdf.name} was built with other settings -> rebuilding once")
    blocks, _ = parse_pdf(pdf)
    chunks = chunk_blocks(blocks)
    del blocks
    if not chunks:
        raise RuntimeError("No text extracted (scanned PDF? run OCR first, e.g. ocrmypdf)")
    if EMBED_ONLY_TECH_CHUNKS:
        kept = [c for c in chunks if TECH_RE.search(c.text) or TECH_RE.search(c.section or "")]
        if len(kept) >= 40:
            print(f"[filter] keeping {len(kept)} of {len(chunks)} chunks that mention technology topics")
            chunks = kept
            for i, c in enumerate(chunks):
                c.id = i
        else:
            print(f"[filter] only {len(kept)} tech chunks found -> keeping all {len(chunks)} chunks")
    for c in chunks:
        c.company = company
    idx = HybridIndex(chunks, embedder, reranker).build()
    idx.save(d)
    meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    meta["tech_only"] = want_tech
    (d / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return idx


def retrieve_for_theme(idx: HybridIndex, spec: dict) -> list:
    """Retrieve for every query of the theme, keep RELEVANCE order (round-robin merge), no page-neighbour padding."""
    lists = []
    old = CFG["neighbor_expand"]
    CFG["neighbor_expand"] = False
    try:
        for q in spec["queries"]:
            lists.append(idx.retrieve(q, hint_keys=spec["hints"], top_k=TOP_K_PER_QUERY))
    finally:
        CFG["neighbor_expand"] = old
    merged, seen = [], set()
    for rank in range(max(len(l) for l in lists)):
        for l in lists:
            if rank < len(l) and l[rank].id not in seen:
                seen.add(l[rank].id)
                merged.append(l[rank])
    return merged


def clean_cell(raw: str, allowed_pages: set) -> str:
    t = _strip_think(raw)
    t = re.sub(r"(?im)^\s*evidence used:.*$", "", t)
    t = re.sub(r"(?im)^\s*_?extractive answer.*$", "", t)
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    if not lines or re.fullmatch(r"(?i)\W*not disclosed\W*", " ".join(lines)):
        return "Not disclosed"
    out, seen = [], set()
    for l in lines:
        l = re.sub(r"^(?:[-*\u2022]+|\d+[.)])\s*", "", l).strip()
        if len(l) < 8 or re.fullmatch(r"(?i)\W*not disclosed\W*", l):
            continue

        def _check(m):
            nums = [int(x) for x in re.findall(r"\d+", m.group(0))]
            if nums and all(n in allowed_pages for n in nums):
                return m.group(0)
            return m.group(0)[:-1] + " \u26a0)"          # cited page was not in the evidence -> flag

        l = CITE_RE.sub(_check, l)
        if not CITE_RE.search(l):
            l += " (p.? \u26a0)"                          # bullet without any page citation -> flag
        key = re.sub(r"\W+", "", l.lower())[:80]
        if key in seen:
            continue
        seen.add(key)
        out.append("\u2022 " + l)
    return "\n".join(out[:MAX_BULLETS]) if out else "Not disclosed"


def extract_cell(idx: HybridIndex, llm: LLMSpec, company: str, theme: str, spec: dict) -> dict:
    chunks = retrieve_for_theme(idx, spec)
    context = build_context(chunks, CONTEXT_CHARS)
    used_pages = {int(x) for x in re.findall(r"--- EXCERPT \(.*?, page (\d+)", context)}
    prompt = CELL_TEMPLATE.format(company=company, theme=theme, focus=spec["focus"],
                                  max_bullets=MAX_BULLETS, context=context)
    raw = llm_complete(llm, CELL_SYSTEM, prompt, max_tokens=2000)
    return {
        "answer": clean_cell(raw, used_pages),
        "pages_retrieved": sorted(used_pages),
        "evidence": [{"page": c.page, "kind": c.kind, "section": c.section[:80], "preview": c.text[:300]}
                     for c in chunks if c.page in used_pages],
    }

print("Helpers ready.")

Helpers ready.


## 11b. Read the company list and match each name to its PDF
Creates `match_report.csv` so you can see which companies found a PDF. Unmatched ones are skipped (fix with `MANUAL_PDF_MAP`).

In [16]:
import difflib
import pandas as pd

# ---------- 1. read the company list ----------
def read_company_list(path) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Company list not found: {p.resolve()}  -> fix COMPANY_CSV")
    if p.suffix.lower() in (".xlsx", ".xls"):
        raw = pd.read_excel(p, header=None, dtype=str)
    else:
        raw = pd.read_csv(p, header=None, dtype=str, encoding="utf-8-sig", encoding_errors="replace")
    name = raw[0].fillna("").astype(str).str.strip()
    sector = raw[11].fillna("").astype(str).str.strip() if raw.shape[1] > 11 else pd.Series("", index=raw.index)
    sector = sector.replace("", np.nan).ffill().fillna("")        # sector is written only on the first company of each block
    df = pd.DataFrame({"Company": name, "Sector": sector})
    df = df[(df["Company"] != "") & (df["Company"].str.lower() != "company name")]
    df = df.drop_duplicates("Company").reset_index(drop=True)
    df.insert(0, "No.", range(1, len(df) + 1))                   # <- START_COMPANY / END_COMPANY refer to this number
    return df

# ---------- 2. name normalisation + similarity ----------
_NOISE = {"ltd", "limited", "pvt", "private", "inc", "corp", "corporation", "co", "the", "annual", "report", "reports",
          "ar", "integrated", "final", "web", "print", "of_", "fy"}

def norm_name(s: str) -> str:
    s = s.lower().replace("&", " and ")
    s = re.sub(r"\.pdf$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    toks = [t for t in s.split() if t not in _NOISE and not re.fullmatch(r"(fy)?\d{2,8}", t)]
    return " ".join(toks)

def name_score(a: str, b: str) -> float:
    na, nb = norm_name(a), norm_name(b)
    if not na or not nb:
        return 0.0
    if na == nb:
        return 1.0
    ta, tb = set(na.split()), set(nb.split())
    jac = len(ta & tb) / len(ta | tb)
    seq = difflib.SequenceMatcher(None, na, nb).ratio()
    return round(0.5 * seq + 0.5 * jac, 3)

# ---------- 3. match company -> PDF (mutual-best, so "Bank of India" can never steal "State Bank of India") ----------
def match_companies(comp_df: pd.DataFrame, pdf_paths: list) -> pd.DataFrame:
    pdf_by_name = {p.name: p for p in pdf_paths}
    comps = comp_df["Company"].tolist()
    S = np.array([[name_score(c, p.stem) for p in pdf_paths] for c in comps])
    best_pdf_for_comp = S.argmax(axis=1)
    best_comp_for_pdf = S.argmax(axis=0)
    rows = []
    for i, c in enumerate(comps):
        j = int(best_pdf_for_comp[i])
        score, pdf = float(S[i, j]), pdf_paths[j]
        status = "NO PDF FOUND"
        if c in MANUAL_PDF_MAP and MANUAL_PDF_MAP[c] in pdf_by_name:
            pdf, score, status = pdf_by_name[MANUAL_PDF_MAP[c]], 1.0, "MATCHED (manual)"
        elif score >= MATCH_THRESHOLD and int(best_comp_for_pdf[j]) == i:
            status = "MATCHED"
        elif score >= MATCH_THRESHOLD:
            status = "NO PDF FOUND (closest PDF belongs to another company)"
        rows.append({"No.": comp_df.iloc[i]["No."], "Company": c, "Sector": comp_df.iloc[i]["Sector"],
                     "Status": status, "PDF File": pdf.name if status.startswith("MATCHED") else "",
                     "Score": round(score, 3), "Closest PDF (for manual map)": pdf.name})
    return pd.DataFrame(rows)

company_df = read_company_list(COMPANY_CSV)
all_pdfs = sorted([p for p in Path(REPORTS_DIR).iterdir() if p.suffix.lower() == ".pdf"], key=lambda p: p.name.lower())
if not all_pdfs:
    raise FileNotFoundError(f"No PDFs found in {Path(REPORTS_DIR).resolve()} -> fix REPORTS_DIR")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
match_df = match_companies(company_df, all_pdfs)
match_df.to_csv(Path(OUTPUT_DIR) / "match_report.csv", index=False, encoding="utf-8-sig")

n_ok = match_df["Status"].str.startswith("MATCHED").sum()
print(f"{len(company_df)} companies in list | {len(all_pdfs)} PDFs in folder | {n_ok} matched | {len(match_df) - n_ok} without PDF")
print(f"Full match report saved: {(Path(OUTPUT_DIR) / 'match_report.csv').resolve()}")
display(match_df.iloc[max(0, START_COMPANY - 1):END_COMPANY][["No.", "Company", "Status", "PDF File", "Score"]])

359 companies in list | 433 PDFs in folder | 326 matched | 33 without PDF
Full match report saved: D:\annual_report_rag_8GB\annual_report_rag_8GB\tech_matrix_output\match_report.csv


,No.,Company,Status,PDF File,Score
0,1,Life Insurance Corporation of India,MATCHED,Life_Insurance_Corporation_of_India_Annual_Rep...,1.0
1,2,Kotak Mahindra Bank Ltd.,MATCHED,Kotak Mahindra_Bank_ltd.pdf,1.0
2,3,Axis Bank Ltd.,MATCHED,Axis Bank Limited.pdf,1.0
3,4,Bajaj Finserv Ltd.,MATCHED,Bajaj Finserv Limited.pdf,1.0
4,5,Union Bank of India,MATCHED,Union Bank of India.pdf,1.0
...,...,...,...,...,...
354,355,Oil India Ltd.,MATCHED,Oil India Limited.pdf,1.0
355,356,Aegis Logistics Ltd.,MATCHED,Aegis Logistics Limited.pdf,1.0
356,357,Petronet LNG Ltd.,MATCHED,Petronet LNG Limited.pdf,1.0
357,358,Aegis Vopak Terminals Ltd.,MATCHED,Aegis Vopak Terminals Limited.pdf,1.0


## 11c. Excel writer (append mode)
One Excel for all companies. Re-running a company replaces its row; other companies are untouched.

In [14]:
import os
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

ILLEGAL_EXCEL_CHARS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")
def clean_excel_value(v):
    return ILLEGAL_EXCEL_CHARS.sub("", v) if isinstance(v, str) else v

PARAMS = list(THEMES)
MATRIX_COLS = ["No.", "Company", "Sector", "PDF File"] + PARAMS
EVID_COLS = ["No.", "Company", "Parameter", "Page", "Type", "Section", "Excerpt (first 300 chars)"]
XLSX_PATH = Path(OUTPUT_DIR) / EXCEL_FILE

def companies_in_excel() -> set:
    if not XLSX_PATH.exists():
        return set()
    try:
        return set(pd.read_excel(XLSX_PATH, sheet_name="Tech Matrix", usecols=["Company"])["Company"].astype(str))
    except Exception:
        return set()

def _format_sheets(xw):
    ws = xw.sheets["Tech Matrix"]
    fill = PatternFill("solid", start_color="1F3864")
    for c in ws[1]:
        c.font, c.fill = Font(bold=True, color="FFFFFF"), fill
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for j, w in enumerate([6, 30, 20, 34] + [60] * len(PARAMS), 1):
        ws.column_dimensions[get_column_letter(j)].width = w
    for row in ws.iter_rows(min_row=2):
        row[1].font = Font(bold=True)
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "C2"
    we = xw.sheets["Evidence"]
    for j, w in enumerate([6, 28, 28, 7, 8, 30, 110], 1):
        we.column_dimensions[get_column_letter(j)].width = w
    for row in we.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    we.freeze_panes = "A2"

def save_company_to_excel(no, company, sector, pdf_name, cells: dict):
    """Adds (or replaces) ONE company in the single Excel. Existing companies are kept -> ranges append."""
    new_row = {"No.": int(no), "Company": company, "Sector": sector, "PDF File": pdf_name}
    new_row.update({p: cells.get(p, {}).get("answer", "(not run)") for p in PARAMS})
    new_ev = [{"No.": int(no), "Company": company, "Parameter": p, "Page": e["page"], "Type": e["kind"],
               "Section": e["section"], "Excerpt (first 300 chars)": e["preview"]}
              for p in PARAMS for e in cells.get(p, {}).get("evidence", [])]

    if XLSX_PATH.exists():
        old_m = pd.read_excel(XLSX_PATH, sheet_name="Tech Matrix", dtype=object)
        old_e = pd.read_excel(XLSX_PATH, sheet_name="Evidence", dtype=object)
        old_m = old_m[old_m["Company"] != company]
        old_e = old_e[old_e["Company"] != company]
    else:
        old_m, old_e = pd.DataFrame(columns=MATRIX_COLS), pd.DataFrame(columns=EVID_COLS)

    matrix = pd.concat([old_m, pd.DataFrame([new_row])], ignore_index=True)
    matrix["No."] = matrix["No."].astype(int)
    matrix = matrix.sort_values("No.").reindex(columns=MATRIX_COLS)
    evid = pd.concat([old_e, pd.DataFrame(new_ev, columns=EVID_COLS)], ignore_index=True)
    evid = evid.reindex(columns=EVID_COLS)
    evid["_o"] = evid["No."].astype(int)
    evid = evid.sort_values("_o", kind="stable").drop(columns="_o")

    tmp = XLSX_PATH.with_name("~tmp_" + XLSX_PATH.name)
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        matrix.map(clean_excel_value).to_excel(xw, sheet_name="Tech Matrix", index=False) if hasattr(matrix, "map") \
            else matrix.applymap(clean_excel_value).to_excel(xw, sheet_name="Tech Matrix", index=False)
        evid.map(clean_excel_value).to_excel(xw, sheet_name="Evidence", index=False) if hasattr(evid, "map") \
            else evid.applymap(clean_excel_value).to_excel(xw, sheet_name="Evidence", index=False)
        _format_sheets(xw)
    try:
        os.replace(tmp, XLSX_PATH)
    except PermissionError:
        raise PermissionError(f"Close '{XLSX_PATH.name}' in Excel and run again (results are safe in cell_cache.json).") from None
print("Excel helper ready ->", XLSX_PATH.resolve())

Excel helper ready -> D:\annual_report_rag_8GB\annual_report_rag_8GB\tech_matrix_output\tech_matrix6_all.xlsx


## 11d. Page-diverse retrieval + sector-aware prompt
Paste/run this AFTER Section 11 (it replaces `retrieve_for_theme`, `CELL_SYSTEM`, `CELL_TEMPLATE` and `extract_cell`).

In [15]:
# ==== 11d. Page-diverse retrieval + sector-aware prompt (overrides the same-named functions from Section 11) ====
MAX_CHUNKS_PER_PAGE = 2        # at most this many chunks from one page -> evidence comes from many different pages

def _norm_sector(s: str) -> str:
    return re.sub(r"[^a-z]", "", str(s).lower())

def sector_of(company: str) -> str:
    for nm in ("match_df", "company_df"):
        df = globals().get(nm)
        if df is not None:
            r = df.loc[df["Company"] == company, "Sector"]
            if len(r):
                return str(r.iloc[0])
    return ""

def sector_profile(company: str) -> dict:
    return SECTOR_PROFILES.get(_norm_sector(sector_of(company)), SECTOR_DEFAULT)

def theme_queries(theme: str, spec: dict, prof: dict) -> list:
    qs = list(spec["queries"])
    if theme in SECTOR_QUERY_THEMES and prof.get("terms"):
        qs.append(f"{spec['queries'][0]} Sector-specific examples: {prof['terms']}")
    return qs

def retrieve_for_theme(idx, spec: dict, prof: dict = None, theme: str = "") -> list:
    """Run every query of the theme, merge round-robin by relevance, but never take more than
    MAX_CHUNKS_PER_PAGE chunks from one page (stops one 'Technology' chapter from filling the whole context)."""
    qs = theme_queries(theme, spec, prof or SECTOR_DEFAULT)
    lists, old = [], CFG["neighbor_expand"]
    CFG["neighbor_expand"] = False
    try:
        for q in qs:
            lists.append(idx.retrieve(q, hint_keys=spec["hints"], top_k=TOP_K_PER_QUERY))
    finally:
        CFG["neighbor_expand"] = old
    merged, seen, per_page = [], set(), {}
    for rank in range(max((len(l) for l in lists), default=0)):
        for l in lists:
            if rank < len(l):
                c = l[rank]
                if c.id in seen or per_page.get(c.page, 0) >= MAX_CHUNKS_PER_PAGE:
                    continue
                seen.add(c.id)
                per_page[c.page] = per_page.get(c.page, 0) + 1
                merged.append(c)
    return merged

CELL_SYSTEM = (
    "You are a meticulous technology-intelligence analyst who reads Indian listed-company annual reports and fills a comparison table. "
    "You use ONLY the report excerpts supplied, never outside knowledge, never guesses, and never invented page numbers. "
    "You are thorough: you capture every relevant initiative, system, partner, amount and metric that appears in the excerpts. "
    "Every bullet ends with the page citation of the excerpt it came from, like (p.34)."
)

CELL_TEMPLATE = """Fill ONE cell of a technology-intelligence table for the company "{company}" (sector: {sector}).
Parameter: {theme}
What to capture: {focus}

SECTOR CONTEXT
In this sector, technology typically includes: {lens}
R&D note for this sector: {rnd}
Count sector-specific technology even when the words "digital" or "technology" are not used (for example: {examples}).
Do NOT count ordinary products, sales or financial results that are not technology.

CONTEXT (excerpts from DIFFERENT parts of the annual report; each is labelled with its PDF page)
--------------------------------------------------------------------------------------------
{context}

RULES
-----
1. Use ONLY the CONTEXT. No outside knowledge, no guessing.
2. Output ONLY bullet lines, each starting with "- ". Up to {max_bullets} bullets, most important first. No headings, no intro, no closing remarks.
3. Be thorough: cover EVERY distinct relevant initiative, system, partner, amount or metric found in the CONTEXT, drawing on as many different pages as possible. Never repeat the same point. Prefer specific facts (names, numbers, dates) over general statements; if only general statements exist, keep the one or two most informative.
4. Each bullet is one complete sentence of about 20-50 words: WHAT it is + HOW it is used or the result + named PARTNER/VENDOR + AMOUNT/metric, whichever of these the CONTEXT states. End with the page in brackets, e.g. "(p.34)". If a fact is on two pages write "(p.34, p.36)". Use the page number from the excerpt header only.
5. Keep exact names of technologies, platforms, vendors, partners, programmes and subsidiaries verbatim. Keep amounts exactly as written with currency and unit (e.g. "Rs 1,200 crore", "USD 45 million"). Never invent or convert a number.
6. A partial answer is fine: write what the CONTEXT supports. Output exactly "Not disclosed" ONLY if the CONTEXT contains nothing relevant to this parameter at all.
7. Do NOT include for this parameter: {avoid}. Do not repeat an item that clearly belongs to another parameter (for example, put AI items under AI unless they are the main point here). Ignore anything that is not about technology.
"""

def extract_cell(idx, llm, company: str, theme: str, spec: dict) -> dict:
    prof = sector_profile(company)
    chunks = retrieve_for_theme(idx, spec, prof, theme)
    context = build_context(chunks, CONTEXT_CHARS)
    used_pages = {int(x) for x in re.findall(r"--- EXCERPT \(.*?, page (\d+)", context)}
    prompt = CELL_TEMPLATE.format(company=company, sector=prof["name"], theme=theme, focus=spec["focus"],
                                  lens=prof["lens"], rnd=prof["rnd"], examples=prof["examples"], avoid=spec.get("avoid", "non-technology items"),
                                  max_bullets=MAX_BULLETS, context=context)
    raw = llm_complete(llm, CELL_SYSTEM, prompt, max_tokens=3000)
    return {
        "answer": clean_cell(raw, used_pages),
        "pages_retrieved": sorted(used_pages),
        "evidence": [{"page": c.page, "kind": c.kind, "section": c.section[:80], "preview": c.text[:300]}
                     for c in chunks if c.page in used_pages],
    }

print(f"11d ready | sector profiles: {len(SECTOR_PROFILES)} | settings in use: TOP_K_PER_QUERY={TOP_K_PER_QUERY}, "
      f"CONTEXT_CHARS={CONTEXT_CHARS}, MAX_BULLETS={MAX_BULLETS}, MAX_CHUNKS_PER_PAGE={MAX_CHUNKS_PER_PAGE}")
if CONTEXT_CHARS < 12000 or TOP_K_PER_QUERY < 5:
    print("TIP: for fuller answers set TOP_K_PER_QUERY = 5 and CONTEXT_CHARS = 14000 in Section 9 (if Groq limits allow).")

11d ready | sector profiles: 11 | settings in use: TOP_K_PER_QUERY=10, CONTEXT_CHARS=14000, MAX_BULLETS=20, MAX_CHUNKS_PER_PAGE=2


In [16]:
for _n in ["get_index", "clean_cell", "build_context", "extract_cell", "retrieve_for_theme",
           "sector_profile", "SECTOR_PROFILES", "THEMES", "save_company_to_excel",
           "companies_in_excel", "match_df", "parse_llm_spec", "llm_complete", "RateLimitStop"]:
    print(f"{_n:24s}", "OK" if _n in globals() else "MISSING  <-- run the cell that defines it")

get_index                OK
clean_cell               OK
build_context            OK
extract_cell             OK
retrieve_for_theme       OK
sector_profile           OK
SECTOR_PROFILES          OK
THEMES                   OK
save_company_to_excel    OK
companies_in_excel       OK
match_df                 OK
parse_llm_spec           OK
llm_complete             OK
RateLimitStop            OK


## 12. Run the selected companies × 15 parameters
For each company in the range: PDF → chunk → embed → retrieve → LLM → 15 cells → **saved/appended to the one Excel right after each company** (safe to stop and re-run).

In [ ]:
import gc
out_dir = Path(OUTPUT_DIR)
CACHE_FILE = out_dir / "cell_cache.json"
cache = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else {}

if START_COMPANY < 1 or END_COMPANY < START_COMPANY:
    raise ValueError("Use START_COMPANY >= 1 and END_COMPANY >= START_COMPANY")
selected = match_df[(match_df["No."] >= START_COMPANY) & (match_df["No."] <= END_COMPANY)]
print(f"Running companies {START_COMPANY} to {END_COMPANY}  ({len(selected)} companies)  |  LLM = {LLM}")

llm = parse_llm_spec(LLM)
if llm.provider == "groq" and not llm.api_key:
    raise RuntimeError("GROQ_API_KEY is not set - run the Section 9b cell first, then re-run Section 9's settings cell is NOT needed, just re-run THIS cell.")
if llm.provider == "gemini" and not llm.api_key:
    raise RuntimeError("GEMINI_API_KEY is not set.")
if llm.provider == "ollama":
    avail = llm_list_models(llm)
    if avail and str(avail[0]).startswith("<could not"):
        raise RuntimeError("Ollama is not reachable - start it and run: ollama pull " + llm.model)

already = companies_in_excel()
embedder = reranker = None
t_start = time.time()
done, skipped, stop_all = [], [], False

for k, (_, row) in enumerate(selected.iterrows(), 1):
    if stop_all:
        break
    no, company, sector = int(row["No."]), row["Company"], row["Sector"]
    print("\n" + "=" * 100 + f"\n[{k}/{len(selected)}] #{no} {company}")
    if not row["Status"].startswith("MATCHED"):
        print("   SKIPPED - no annual-report PDF matched (see match_report.csv / use MANUAL_PDF_MAP)")
        skipped.append((no, company, "no PDF"))
        continue
    if company in already and not FORCE_RERUN:
        print("   SKIPPED - already in the Excel (set FORCE_RERUN=True to redo)")
        continue
    pdf = Path(REPORTS_DIR) / row["PDF File"]
    todo = [t for t in THEMES if FORCE_RERUN or f"{company}||{t}" not in cache
            or str(cache[f"{company}||{t}"]["answer"]).startswith("ERROR")]
    print(f"   PDF: {pdf.name}  | {len(todo)} of 15 parameters to extract")
    idx = None
    try:
        if todo:
            if embedder is None:
                embedder, reranker = Embedder(), Reranker(enabled=USE_RERANKER)
            idx = get_index(pdf, company, embedder, reranker)
    except Exception as e:
        print(f"   ERROR could not index {pdf.name}: {e}")
        skipped.append((no, company, f"index error: {e}"))
        gc.collect()
        continue
    for ti, theme in enumerate(todo, 1):
        t0 = time.time()
        try:
            cell = extract_cell(idx, llm, company, theme, THEMES[theme])
            cache[f"{company}||{theme}"] = cell
            nb_ = 0 if cell["answer"] == "Not disclosed" else cell["answer"].count("\u2022")
            print(f"   ({ti}/{len(todo)}) {theme:30s} {nb_} bullets  [{time.time() - t0:.0f}s]")
        except RateLimitStop as e:
            print(f"\n   STOP: {e}")
            stop_all = True
            break
        except Exception as e:
            print(f"   ({ti}/{len(todo)}) {theme:30s} ERROR {str(e)[:140]}")
            cache[f"{company}||{theme}"] = {"answer": f"ERROR: {e}", "pages_retrieved": [], "evidence": []}
            if "HTTP 401" in str(e) or "HTTP 403" in str(e) or "HTTP 404" in str(e):
                print("   -> Not retryable (check API key / network / model name). Stopping.")
                stop_all = True
                break
        CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
        if llm.provider != "mock":
            time.sleep(LLM_PAUSE)
    CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
    if stop_all:
        print("   Run stopped before this company was complete -> NOT saved to Excel. Fix the issue above and re-run Section 12.")
        skipped.append((no, company, "stopped"))
        break
    cells = {t: cache.get(f"{company}||{t}", {}) for t in THEMES}
    bad = [t for t in THEMES if (not cells[t]) or str(cells[t].get("answer", "")).startswith("ERROR")]
    if bad:
        print(f"   NOT saved to Excel: {len(bad)} parameter(s) incomplete ({', '.join(bad[:4])}...). Re-run Section 12 - finished ones are cached.")
        skipped.append((no, company, "incomplete"))
    else:
        save_company_to_excel(no, company, sector, pdf.name, cells)       # saved/appended after EACH company
        done.append(company)
        print(f"   SAVED to {XLSX_PATH.name}")
    idx = None
    gc.collect()

print(f"\nFinished in {(time.time() - t_start) / 60:.1f} min | saved: {len(done)} | skipped/incomplete: {len(skipped)}")
for sk in skipped:
    print("   ", sk)
print("Excel:", XLSX_PATH.resolve())

Running companies 202 to 202  (1 companies)  |  LLM = groq:openai/gpt-oss-120b

[1/1] #202 Indus Towers
   PDF: Indus_Towers_Annual_Report.pdf  | 15 of 15 parameters to extract


d:\annual_report_rag_8GB\annual_report_rag_8GB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[embed] model=BAAI/bge-small-en-v1.5 device=cpu
[rerank] model=cross-encoder/ms-marco-MiniLM-L-6-v2 device=cpu
   ... parsed 50/258 pages (123s)


In [ ]:
try:
    print(llm_complete(parse_llm_spec(LLM), "Reply with OK", "Say OK", max_tokens=300, retries=1))
except Exception as e:
    print(str(e)[:600])

[llm] retry 1/1 after error: HTTP 429: {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4x19gh1e76b21jvf150v88p` service tier `on_demand` on t (sleep 61.116s)
LLM call failed: HTTP 429: {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m4x19gh1e76b21jvf150v88p` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199759, Requested 379. Please try again in 59.616s. Need more tokens? Upgrade to Dev Tier today at https://con


## 13. View the combined Excel
Output file: `OUTPUT_DIR/tech_matrix_all.xlsx` — sheet **Tech Matrix** (one row per company, 15 parameter columns) and sheet **Evidence** (excerpts/pages behind every cell). Close the file in Excel before running Section 12.

In [ ]:
# View the combined Excel (all companies processed so far)
final = pd.read_excel(XLSX_PATH, sheet_name="Tech Matrix")
print(f"{len(final)} companies in {XLSX_PATH.name}:  No. {final['No.'].min()} ... {final['No.'].max()}")
display(final.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left",
                                      "vertical-"
                                      "align": "top", "min-width": "320px"}))

## 14. Inspect one cell (debug)
If a cell looks thin or wrong, look at the exact excerpts and pages behind it, then adjust that theme's `focus` / `queries` in Section 10 (or raise `TOP_K_PER_QUERY`) and re-run Sections 12-13 with `FORCE_RERUN = True`.

In [ ]:
COMPANY, THEME = list(cache)[0].split("||")        # <-- or type e.g. COMPANY, THEME = "ICICI Bank Ltd.", "AI"
cell = cache[f"{COMPANY}||{THEME}"]
print(f"{COMPANY} / {THEME}\n" + "-" * 80)
print(cell["answer"])
print("\nPages given to the LLM:", cell["pages_retrieved"])
for ev in cell["evidence"][:6]:
    print(f"\n[p.{ev['page']} | {ev['kind']} | {ev['section']}]\n{ev['preview']}")

ICICI Bank Ltd. / Current Technology Adoption
--------------------------------------------------------------------------------
• The Bank operates nine Digital Banking Units (DBUs) providing self‑service and assisted digital banking, opening 1,025 accounts, issuing 882 credit cards, disbursing 247 loans, processing 116,566 financial and 23,224 non‑financial transactions, and conducting 255 digital‑awareness camps in FY‑2026 (p.161).
• ICICI Bank’s DigiEase digital platform supports end‑to‑end business‑banking workflows, including mobile flow, supervisory dashboard, Credit Guarantee Fund Trust coverage and planned integration with the Account Aggregator framework (p.131).
• Internet Banking employs QR‑based login for seamless, secure access, while the iMobile app offers a Smart Lock feature that lets customers temporarily disable banking channels to prevent unauthorized use (p.125).
• Business centres use biometric authentication, tab‑banking terminals, digital service kiosks, iCRM and 

## Troubleshooting
| Problem | Fix |
|---|---|
| `HTTP 401` | Wrong/missing Groq key - re-run Section 9b with a fresh key. |
| `HTTP 403 / 1010` | Network/VPN blocked by Cloudflare - turn off VPN or use a phone hotspot. |
| `HTTP 404` / model not found | Fix the `LLM` line (model names are listed in console.groq.com). |
| `STOP: ... rate/daily limit` | Free-tier token limit reached. Wait the time shown (or switch to `groq:openai/gpt-oss-20b` / Gemini) and re-run Section 12 - finished cells are cached. |
| Out-of-memory / laptop freezes | Close Chrome/Teams, restart the kernel, run ONE company at a time (START = END). |
| Many cells say `Not disclosed` | Raise `TOP_K_PER_QUERY` to 8-10, or set `EMBED_ONLY_TECH_CHUNKS = False` and `REBUILD_INDEX = True` (slower). |
| `No text extracted` | Scanned PDF - OCR it first (`ocrmypdf in.pdf out.pdf`). |
| `PermissionError` on the Excel | Close `tech_matrix_all.xlsx` in Excel and re-run. |
| Company shows `NO PDF FOUND` | Rename the PDF to the company name or use `MANUAL_PDF_MAP` (see `match_report.csv`). |